In [ ]:
import warnings, logging, os
warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"
logging.getLogger().setLevel(logging.ERROR)

!pip install -q --no-warn-conflicts \
    streamlit==1.41.0 \
    pyngrok \
    folium==0.16.0 \
    streamlit-folium==0.20.0 \
    plotly \
    xgboost==1.7.6 \
    shap==0.46.0 \
    scikit-learn \
    pandas \
    numpy \
    pillow \
    reportlab 2>/dev/null

import logging
logging.getLogger().setLevel(logging.ERROR)
print("✅ All done! Run Cell 2 now 👉")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.4/23.4 MB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.0/100.0 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 326.7/326.7 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.3/200.3 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.9/543.9 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 57.4 MB/s eta 0:00:00
✅ All done! Run Cell 2 now 👉


In [ ]:
%%writefile app.py
import os
os.environ["STREAMLIT_SERVER_ENABLE_STATIC_SERVING"] = "true"
os.environ["STREAMLIT_BROWSER_GATHER_USAGE_STATS"]   = "false"
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import folium
from streamlit_folium import st_folium
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
import shap
from PIL import Image
import math, time, warnings, os
warnings.filterwarnings("ignore")

# ── Page Config ──────────────────────────────────────────────
st.set_page_config(
    page_title="PropAura — Tamil Nadu Market Intelligence",
    page_icon="🏘️",
    layout="wide",
    initial_sidebar_state="expanded"
)
# ── Force dark theme via config ──────────────────────────────
st.markdown("""
<style>
/* ── White background everywhere ── */
html, body, .stApp,
div[data-testid="stAppViewContainer"],
div[data-testid="stHeader"],
section[data-testid="stSidebar"],
.main, .block-container {
    background-color: #ffffff !important;
    color: #1a1a1a !important;
}

/* ── All text black ── */
p, span, div, label, h1, h2, h3, h4, h5, h6,
li, td, th, strong, em, small,
[data-testid="stWidgetLabel"] p,
[data-testid="stWidgetLabel"],
.stMarkdown p, .stMarkdown h4,
[data-testid="stText"] {
    color: #1a1a1a !important;
}

/* ── Sidebar white ── */
[data-testid="stSidebar"] {
    background-color: #f5f5f5 !important;
    border-right: 1px solid #dddddd !important;
}
[data-testid="stSidebar"] * {
    color: #1a1a1a !important;
}

/* ── Selectbox ── */
[data-baseweb="select"] div,
[data-baseweb="select"] input,
[data-baseweb="select"] span {
    background-color: #f5f5f5 !important;
    color: #1a1a1a !important;
    border-color: #cccccc !important;
}
[data-baseweb="menu"] li,
[data-baseweb="menu"] div,
[data-baseweb="popover"] li {
    background-color: #ffffff !important;
    color: #1a1a1a !important;
}
[data-baseweb="menu"] li:hover {
    background-color: #e8f0fe !important;
}

/* ── Inputs ── */
input[type="number"], input[type="text"] {
    background-color: #f5f5f5 !important;
    color: #1a1a1a !important;
    border: 1px solid #cccccc !important;
    border-radius: 8px !important;
}

/* ── Multiselect tags ── */
[data-baseweb="tag"] {
    background-color: #e8f0fe !important;
    border-radius: 4px !important;
    border: 1.5px solid #1a73e8 !important;
}
[data-baseweb="tag"] span,
[data-baseweb="tag"] div,
[data-baseweb="tag"] * {
    color: #1a73e8 !important;
    font-weight: 700 !important;
}

/* ── Multiselect dropdown options ── */
[data-baseweb="menu"] ul li,
[data-baseweb="menu"] ul li span,
[data-baseweb="menu"] ul li div,
[data-baseweb="popover"] ul li,
[data-baseweb="popover"] ul li span,
[role="option"],
[role="option"] span,
[role="option"] div,
[role="listbox"] li,
[role="listbox"] li span {
    color: #1a1a1a !important;
    background-color: #ffffff !important;
}

[role="option"]:hover,
[role="option"][aria-selected="true"] {
    background-color: #e8f0fe !important;
    color: #1a1a1a !important;
}

/* ── Buttons ── */
.stButton > button {
    background: linear-gradient(135deg, #1a73e8, #1557b0) !important;
    color: #ffffff !important;
    border: none !important;
    border-radius: 8px !important;
    font-weight: 700 !important;
    padding: .55rem 1.6rem !important;
}
.stButton > button:hover {
    transform: translateY(-2px) !important;
    box-shadow: 0 6px 20px rgba(26,115,232,.4) !important;
}

/* ── KPI Cards ── */
.kpi {
    background: #f8f9fa !important;
    border: 1px solid #dddddd !important;
    border-radius: 12px;
    padding: 20px 22px;
    text-align: center;
    transition: .2s;
}
.kpi:hover {
    transform: translateY(-3px);
    box-shadow: 0 8px 28px rgba(0,0,0,0.12);
}
.kpi-lbl {
    font-size: .72rem !important;
    font-weight: 700 !important;
    color: #555555 !important;
    text-transform: uppercase;
    letter-spacing: .9px;
    margin-bottom: 6px;
}
.kpi-val {
    font-size: 1.85rem !important;
    font-weight: 800 !important;
    color: #1a73e8 !important;
    line-height: 1.1;
}
.kpi-sub {
    font-size: .72rem !important;
    color: #2e7d32 !important;
    margin-top: 5px;
}

/* ── Section header ── */
.sec-hdr {
    background: linear-gradient(135deg, #e8f0fe, #f8f9fa);
    border-left: 4px solid #1a73e8;
    border-radius: 8px;
    padding: 14px 20px;
    margin-bottom: 20px;
}
.sec-hdr h2 {
    margin: 0 !important;
    font-size: 1.25rem !important;
    color: #1a1a1a !important;
}
.sec-hdr p {
    margin: 4px 0 0 !important;
    font-size: .8rem !important;
    color: #555555 !important;
}

/* ── Result cards ── */
.res-card {
    background: #f8f9fa !important;
    border: 1px solid #dddddd !important;
    border-radius: 12px;
    padding: 22px;
    margin: 10px 0;
}
.res-card * { color: #1a1a1a !important; }

/* ── Price gradient ── */
.price-big {
    font-size: 2.6rem;
    font-weight: 800;
    background: linear-gradient(90deg, #1a73e8, #2e7d32);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
}

/* ── Badges ── */
.badge {
    display: inline-block;
    padding: 4px 12px;
    border-radius: 20px;
    font-size: .75rem;
    font-weight: 700;
    margin: 3px;
}
.b-blue   { background:#e8f0fe; color:#1a73e8 !important; border:1px solid #1a73e8; }
.b-green  { background:#e6f4ea; color:#2e7d32 !important; border:1px solid #2e7d32; }
.b-red    { background:#fce8e6; color:#c62828 !important; border:1px solid #c62828; }
.b-gold   { background:#fef9e7; color:#f57f17 !important; border:1px solid #f57f17; }
.b-purple { background:#f3e8fd; color:#6a1b9a !important; border:1px solid #6a1b9a; }

/* ── Score bars ── */
.sb-wrap {
    background: #e0e0e0;
    border-radius: 20px;
    height: 10px;
    overflow: hidden;
    margin: 5px 0 12px;
}
.sb-fill {
    height: 100%;
    border-radius: 20px;
    transition: width .9s ease;
}

/* ── Metrics ── */
[data-testid="stMetricValue"] { color: #1a73e8 !important; font-weight: 800 !important; }
[data-testid="stMetricLabel"] { color: #555555 !important; font-weight: 600 !important; }

/* ── Dataframe ── */
[data-testid="stDataFrame"] * { color: #1a1a1a !important; }

hr { border-color: #dddddd !important; }
</style>
""", unsafe_allow_html=True)

# ══════════════════════════════════════════════════════════════
#  HELPERS
# ══════════════════════════════════════════════════════════════
def fmt_inr(v):
    v = int(v)
    if v >= 10_000_000: return f"₹{v/10_000_000:.2f} Cr"
    if v >= 100_000:    return f"₹{v/100_000:.2f} L"
    return f"₹{v:,}"

def kpi_card(label, val, sub="", color="var(--blue)"):
    return f"""<div class="kpi">
      <div class="kpi-lbl">{label}</div>
      <div class="kpi-val" style="color:{color}">{val}</div>
      {'<div class="kpi-sub">'+sub+'</div>' if sub else ''}
    </div>"""

def sec_header(icon, title, sub=""):
    st.markdown(f"""<div class="sec-hdr">
      <h2>{icon} {title}</h2>{'<p>'+sub+'</p>' if sub else ''}
    </div>""", unsafe_allow_html=True)

def score_bar(label, score, maxv=10, color="#58a6ff"):
    pct = max(0, min(100, score/maxv*100))
    st.markdown(f"""<div style="margin-bottom:9px">
      <div style="display:flex;justify-content:space-between;margin-bottom:3px">
        <span style="font-size:.8rem;color:#8b949e">{label}</span>
        <span style="font-size:.8rem;font-weight:600;color:{color}">{score:.1f}/{maxv}</span>
      </div>
      <div class="sb-wrap"><div class="sb-fill" style="width:{pct}%;background:linear-gradient(90deg,{color},{color}88)"></div></div>
    </div>""", unsafe_allow_html=True)

def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    dlat = math.radians(lat2-lat1); dlon = math.radians(lon2-lon1)
    a = math.sin(dlat/2)**2 + math.cos(math.radians(lat1))*math.cos(math.radians(lat2))*math.sin(dlon/2)**2
    return round(R * 2 * math.asin(math.sqrt(a)), 2)

PLOTLY_LAYOUT = dict(
    paper_bgcolor="#ffffff",
    plot_bgcolor="#f8f9fa",
    font_color="#1a1a1a",
    margin=dict(l=12,r=12,t=18,b=12)
)


# ══════════════════════════════════════════════════════════════
#  LOAD DATA
# ══════════════════════════════════════════════════════════════
@st.cache_data
def load_data():
    csv = "/content/Smart_real_estate.csv"
    if not os.path.exists(csv):
        st.error("❌Smart_real_estate.csv not found! Upload it alongside this app.")
        st.stop()
    return pd.read_csv(csv)

df = load_data()

# ══════════════════════════════════════════════════════════════
#  MODEL TRAINING
# ══════════════════════════════════════════════════════════════
@st.cache_resource
def train_model(data):
    d = data.copy()
    for col in ["furnished","property_type","facing","city"]:
        le = LabelEncoder()
        d[col+"_enc"] = le.fit_transform(d[col])

    features = [
        "area_sqft","bedrooms","bathrooms","floor","total_floors","age_years",
        "parking","amenities_score","furnished_enc","property_type_enc",
        "facing_enc","city_enc",
        "dist_city_center_km","dist_school_km","dist_hospital_km",
        "dist_metro_km","dist_highway_km"
    ]
    X = d[features]; y = d["price_inr"]
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
    m = xgb.XGBRegressor(n_estimators=400, learning_rate=0.04, max_depth=6,
                          subsample=0.8, colsample_bytree=0.8,
                          random_state=42, verbosity=0)
    m.fit(Xtr, ytr)
    preds = m.predict(Xte)
    explainer = shap.TreeExplainer(m)
    encoders = {}
    for col in ["furnished","property_type","facing","city"]:
        le2 = LabelEncoder(); le2.fit(data[col])
        encoders[col] = le2
    return m, features, mean_absolute_error(yte,preds), r2_score(yte,preds), explainer, Xte, yte, encoders, d

model, feat_cols, mae, r2v, explainer, X_test, y_test, encoders, df_enc = train_model(df)


# ══════════════════════════════════════════════════════════════
#  SIDEBAR
# ══════════════════════════════════════════════════════════════
with st.sidebar:
    st.markdown("""
    <div style="text-align:center;padding:18px 0 10px">
      <div style="font-size:2.4rem">🏙️</div>
      <div style="font-size:1.2rem;font-weight:700;color:#58a6ff">PropAura</div>
      <div style="font-size:.7rem;color:#8b949e;margin-top:3px">Tamil Nadu Market Intelligence</div>
    </div><hr>""", unsafe_allow_html=True)

    page = st.radio("", [
        "🏠 Smart Dashboard",
        "📍 Geo Intelligence",
        "🏡 Price Prediction",
        "🖼️ Image Intelligence",
        "🧠 Model Explainability",
        "📈 Investment Advisor",
        "🔁 Property Comparator",
        "💸 EMI Calculator",
        "🏗️ Blueprint Generator",
    ], label_visibility="collapsed")

    st.markdown("<hr>", unsafe_allow_html=True)
    st.markdown(f"""
    <div style="font-size:.72rem;color:#8b949e;text-align:center;line-height:1.8">
      Dataset: <b style="color:#58a6ff">{len(df):,} properties</b><br>
      Cities: Chennai · CBE · Madurai · Salem · Tirunelveli · Erode · Vellore · Thanjavur · Tiruppur · Hosur · Nagercoil · Trichy<br>
      Model R²: <b style="color:#3fb950">{r2v:.3f}</b> &nbsp;|&nbsp; MAE: <b>{fmt_inr(mae)}</b><br><br>
      <span style="color:#3fb950">● System Online</span>
    </div>""", unsafe_allow_html=True)


# ══════════════════════════════════════════════════════════════
#  PAGE 1 — SMART DASHBOARD
# ══════════════════════════════════════════════════════════════
if page == "🏠 Smart Dashboard":
    sec_header("🏠", "Smart Dashboard", "Real-time Tamil Nadu property market overview")

    # KPI row
    avg_p   = df["price_inr"].mean()
    avg_psf = (df["price_inr"] / df["area_sqft"]).mean()
    top_loc = df.groupby("locality")["price_inr"].median().idxmax()
    top_city= df.groupby("city")["price_inr"].median().idxmax()

    c1,c2,c3,c4 = st.columns(4)
    c1.markdown(kpi_card("Total Properties", f"{len(df):,}", "Across Tamil Nadu"), unsafe_allow_html=True)
    c2.markdown(kpi_card("Avg Market Price", fmt_inr(avg_p), "All cities"), unsafe_allow_html=True)
    c3.markdown(kpi_card("Avg Price / sqft", f"₹{avg_psf:,.0f}", "Market rate"), unsafe_allow_html=True)
    c4.markdown(kpi_card("Premium City", top_city, "Highest median", color="var(--gold)"), unsafe_allow_html=True)

    st.markdown("<br>", unsafe_allow_html=True)

    # Row 2
    cl, cr = st.columns([3,2])
    with cl:
        st.markdown("#### 📈 Price Trend by Year Listed")
        trend = df.groupby("year_listed")["price_inr"].median().reset_index()
        fig = px.area(trend, x="year_listed", y="price_inr",
                      labels={"price_inr":"Median Price (₹)","year_listed":"Year"},
                      color_discrete_sequence=["#58a6ff"])
        fig.update_traces(fillcolor="rgba(88,166,255,.12)", line_color="#58a6ff")
        fig.update_layout(**PLOTLY_LAYOUT,
            xaxis=dict(showgrid=False,color="#8b949e"),
            yaxis=dict(gridcolor="#30363d",color="#8b949e"))
        st.plotly_chart(fig, use_container_width=True)

    with cr:
        st.markdown("#### 🏙️ Median Price by City")
        city_med = df.groupby("city")["price_inr"].median().sort_values(ascending=True).reset_index()
        fig2 = px.bar(city_med, x="price_inr", y="city", orientation="h",
                      color="price_inr", color_continuous_scale="Blues",
                      labels={"price_inr":"Median Price","city":""})
        fig2.update_layout(**PLOTLY_LAYOUT, coloraxis_showscale=False,
            yaxis=dict(color="#8b949e",tickfont=dict(size=11)),
            xaxis=dict(gridcolor="#30363d",color="#8b949e"))
        st.plotly_chart(fig2, use_container_width=True)

    # Row 3
    ca, cb = st.columns([2,3])
    with ca:
        st.markdown("#### 📐 Area vs Price (by City)")
        fig3 = px.scatter(df.sample(500), x="area_sqft", y="price_inr", color="city",
                          opacity=.65, labels={"area_sqft":"Area (sqft)","price_inr":"Price (₹)","city":"City"})
        fig3.update_layout(**PLOTLY_LAYOUT,
            xaxis=dict(gridcolor="#30363d",color="#8b949e"),
            yaxis=dict(gridcolor="#30363d",color="#8b949e"),
            legend=dict(bgcolor="#1c2128",bordercolor="#30363d"))
        st.plotly_chart(fig3, use_container_width=True)

    with cb:
        st.markdown("#### 🗺️ Live PropAura Property Map")

        # ── Summary stats above map ───────────────────────────
        ms1, ms2, ms3, ms4 = st.columns(4)
        ms1.markdown(kpi_card("Total Properties",
                    f"{len(df):,}", "Mapped below"),
                    unsafe_allow_html=True)
        ms2.markdown(kpi_card("Cities on Map",
                    "12", "Across Tamil Nadu"),
                    unsafe_allow_html=True)
        ms3.markdown(kpi_card("Avg Price",
                    fmt_inr(int(df['price_inr'].mean())),
                    "All cities"),
                    unsafe_allow_html=True)
        ms4.markdown(kpi_card("Price Range",
                    f"{fmt_inr(int(df['price_inr'].min()))} – {fmt_inr(int(df['price_inr'].max()))}",
                    "Min to Max"),
                    unsafe_allow_html=True)

        st.markdown("<br>", unsafe_allow_html=True)

        # ── Map instruction bar ───────────────────────────────
        st.markdown("""
        <div style="background:#e8f0fe;border-radius:8px;
                    padding:10px 16px;margin-bottom:12px;
                    font-size:.83rem;color:#1a3a5c">
          💡 <b>How to use:</b> &nbsp;
          🖱️ <b>Click</b> any dot to see property details &nbsp;|&nbsp;
          🔍 <b>Scroll</b> to zoom in/out &nbsp;|&nbsp;
          🏙️ <b>City labels</b> show city names &nbsp;|&nbsp;
          ⛶ <b>Fullscreen</b> button on top right &nbsp;|&nbsp;
          🔀 <b>Layer switcher</b> to change map style
        </div>""", unsafe_allow_html=True)

        m = folium.Map(
            location=[10.8505, 78.6569],
            zoom_start=7,
            tiles="OpenStreetMap",
            zoom_control=True,
            scrollWheelZoom=True,
            dragging=True,
        )

        from folium.plugins import Fullscreen
        Fullscreen(
            position="topright",
            title="Full Screen",
            title_cancel="Exit Full Screen",
            force_separate_button=True
        ).add_to(m)

        # Google Maps layers
        folium.TileLayer(
            tiles="https://mt1.google.com/vt/lyrs=r&x={x}&y={y}&z={z}",
            attr="Google Maps",
            name="🗺️ Google Maps",
            overlay=False, control=True
        ).add_to(m)
        folium.TileLayer(
            tiles="https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}",
            attr="Google Satellite",
            name="🛰️ Satellite",
            overlay=False, control=True
        ).add_to(m)
        folium.TileLayer(
            tiles="https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}",
            attr="Google Hybrid",
            name="🌍 Hybrid",
            overlay=False, control=True
        ).add_to(m)
        folium.TileLayer(
            "OpenStreetMap",
            name="🗺️ OpenStreetMap",
            overlay=False, control=True
        ).add_to(m)
        folium.LayerControl(position="topright").add_to(m)

        CITY_COLORS_MAP = {
            "Chennai":     "#1a73e8",
            "Coimbatore":  "#2e7d32",
            "Madurai":     "#f57f17",
            "Trichy":      "#c62828",
            "Salem":       "#6a1b9a",
            "Tirunelveli": "#00838f",
            "Erode":       "#e65100",
            "Vellore":     "#1565c0",
            "Thanjavur":   "#4527a0",
            "Tiruppur":    "#558b2f",
            "Hosur":       "#d84315",
            "Nagercoil":   "#00695c",
        }

        CITY_CENTERS_MAP = {
            "Chennai":     (13.0827, 80.2707),
            "Coimbatore":  (11.0168, 76.9558),
            "Madurai":     (9.9252,  78.1198),
            "Trichy":      (10.7905, 78.7047),
            "Salem":       (11.6643, 78.1460),
            "Tirunelveli": (8.7139,  77.7567),
            "Erode":       (11.3410, 77.7172),
            "Vellore":     (12.9165, 79.1325),
            "Thanjavur":   (10.7870, 79.1378),
            "Tiruppur":    (11.1085, 77.3411),
            "Hosur":       (12.7409, 77.8253),
            "Nagercoil":   (8.1833,  77.4119),
        }

        # ── Property dots with rich popups ────────────────────
        sample = df.sample(min(400, len(df)))
        for _, row in sample.iterrows():
            clr = CITY_COLORS_MAP.get(row["city"], "#1a73e8")
            folium.CircleMarker(
                [row["latitude"], row["longitude"]],
                radius=5,
                color=clr,
                fill=True,
                fill_color=clr,
                fill_opacity=0.75,
                weight=1,
                popup=folium.Popup(f"""
                <div style='font-family:Arial;
                            min-width:180px;padding:4px'>
                  <div style='font-weight:700;font-size:13px;
                              color:#1a1a1a;margin-bottom:6px;
                              border-bottom:2px solid {clr};
                              padding-bottom:4px'>
                    📍 {row['locality'].split(',')[0]}
                  </div>
                  <table style='font-size:12px;
                                color:#333;width:100%'>
                    <tr><td>🏙️ City</td>
                        <td><b>{row['city']}</b></td></tr>
                    <tr><td>💰 Price</td>
                        <td><b style='color:#1a73e8'>
                          {fmt_inr(row['price_inr'])}
                        </b></td></tr>
                    <tr><td>🏠 Type</td>
                        <td>{row['property_type']}</td></tr>
                    <tr><td>📐 Area</td>
                        <td>{row['area_sqft']} sqft</td></tr>
                    <tr><td>🛏️ BHK</td>
                        <td>{row['bedrooms']} BHK</td></tr>
                    <tr><td>🛋️ Furnish</td>
                        <td>{row['furnished']}</td></tr>
                    <tr><td>📅 Listed</td>
                        <td>{row['year_listed']}</td></tr>
                  </table>
                </div>""", max_width=220)
            ).add_to(m)

        # ── City labels with stats ────────────────────────────
        for city, (clat, clon) in CITY_CENTERS_MAP.items():
            clr       = CITY_COLORS_MAP[city]
            city_data = df[df["city"] == city]
            avg_p     = fmt_inr(int(city_data["price_inr"].mean()))
            count     = len(city_data)
            folium.Marker(
                [clat, clon],
                popup=folium.Popup(f"""
                <div style='font-family:Arial;
                            min-width:160px;padding:4px'>
                  <div style='font-weight:700;font-size:14px;
                              color:{clr};margin-bottom:6px'>
                    🏙️ {city}
                  </div>
                  <table style='font-size:12px;color:#333'>
                    <tr><td>📊 Listings</td>
                        <td><b>{count}</b></td></tr>
                    <tr><td>💰 Avg Price</td>
                        <td><b style='color:#1a73e8'>
                          {avg_p}
                        </b></td></tr>
                    <tr><td>📍 Localities</td>
                        <td><b>{city_data['locality'].nunique()}</b>
                        </td></tr>
                  </table>
                </div>""", max_width=200),
                icon=folium.DivIcon(
                    html=f"""
                    <div style="
                        background:{clr};
                        color:white;
                        padding:4px 10px;
                        border-radius:14px;
                        font-size:12px;
                        font-weight:700;
                        white-space:nowrap;
                        border:2px solid white;
                        box-shadow:0 2px 8px rgba(0,0,0,0.3);
                        cursor:pointer">
                        {city}
                    </div>""",
                    icon_size=(120, 30),
                    icon_anchor=(60, 15))
            ).add_to(m)

        # ── Legend ────────────────────────────────────────────
        legend_html = """
        <div style='position:fixed;bottom:20px;left:20px;
                    z-index:9999;background:white;
                    border:2px solid #cccccc;
                    border-radius:10px;
                    padding:12px 16px;font-size:11px;
                    box-shadow:0 4px 12px rgba(0,0,0,0.15);
                    max-width:200px'>
          <div style='font-weight:700;margin-bottom:8px;
                      color:#1a1a1a;border-bottom:2px solid #1a73e8;
                      padding-bottom:5px;font-size:12px'>
            🗺️ PropAura City Map
          </div>
          <div style='margin-bottom:6px;color:#555;font-size:10px'>
            🔵 Each dot = 1 property listing<br>
            🏷️ Click dot = property details<br>
            🏙️ Click city label = city stats
          </div>
          <div style='border-top:1px solid #eee;
                      padding-top:6px;margin-top:4px'>
            <div style='font-weight:700;margin-bottom:4px;
                        color:#1a1a1a'>Cities</div>
            <div style='display:grid;
                        grid-template-columns:1fr 1fr;
                        gap:2px 8px'>
              <div><span style='color:#1a73e8;
                font-size:14px'>●</span> Chennai</div>
              <div><span style='color:#2e7d32;
                font-size:14px'>●</span> CBE</div>
              <div><span style='color:#f57f17;
                font-size:14px'>●</span> Madurai</div>
              <div><span style='color:#c62828;
                font-size:14px'>●</span> Trichy</div>
              <div><span style='color:#6a1b9a;
                font-size:14px'>●</span> Salem</div>
              <div><span style='color:#00838f;
                font-size:14px'>●</span> TVL</div>
              <div><span style='color:#e65100;
                font-size:14px'>●</span> Erode</div>
              <div><span style='color:#1565c0;
                font-size:14px'>●</span> Vellore</div>
              <div><span style='color:#4527a0;
                font-size:14px'>●</span> TJR</div>
              <div><span style='color:#558b2f;
                font-size:14px'>●</span> TPR</div>
              <div><span style='color:#d84315;
                font-size:14px'>●</span> Hosur</div>
              <div><span style='color:#00695c;
                font-size:14px'>●</span> NCL</div>
            </div>
          </div>
        </div>"""
        m.get_root().html.add_child(folium.Element(legend_html))
        st_folium(m, width=750, height=480, returned_objects=[])

    # Row 4 — property type & furnished
    cd, ce = st.columns(2)
    with cd:
        st.markdown("#### 🏘️ Property Type Distribution")
        pt = df["property_type"].value_counts().reset_index()
        pt.columns = ["type","count"]
        fig4 = px.pie(pt, names="type", values="count",
                      color_discrete_sequence=["#58a6ff","#3fb950","#e3b341","#a371f7"],
                      hole=0.45)
        fig4.update_layout(**PLOTLY_LAYOUT, showlegend=True,
            legend=dict(bgcolor="#1c2128",bordercolor="#30363d"))
        st.plotly_chart(fig4, use_container_width=True)

    with ce:
        st.markdown("#### 🛋️ Avg Price by Furnishing")
        furn = df.groupby("furnished")["price_inr"].mean().reset_index()
        fig5 = px.bar(furn, x="furnished", y="price_inr",
                      color="furnished",
                      color_discrete_sequence=["#f78166","#e3b341","#3fb950"],
                      labels={"price_inr":"Avg Price (₹)","furnished":""})
        fig5.update_layout(**PLOTLY_LAYOUT, showlegend=False,
            xaxis=dict(color="#8b949e"),
            yaxis=dict(gridcolor="#30363d",color="#8b949e"))
        st.plotly_chart(fig5, use_container_width=True)


# ══════════════════════════════════════════════════════════════
#  PAGE 2 — GEO INTELLIGENCE
# ══════════════════════════════════════════════════════════════
elif page == "📍 Geo Intelligence":
    sec_header("📍", "Geo Intelligence",
               "Real location analytics — schools, hospitals, banks & more via OpenStreetMap")

    import requests as req

    CITY_CENTERS = {
        "Chennai":      (13.0827, 80.2707),
        "Coimbatore":   (11.0168, 76.9558),
        "Madurai":      (9.9252,  78.1198),
        "Trichy":       (10.7905, 78.7047),
        "Salem":        (11.6643, 78.1460),
        "Tirunelveli":  (8.7139,  77.7567),
        "Erode":        (11.3410, 77.7172),
        "Vellore":      (12.9165, 79.1325),
        "Thanjavur":    (10.7870, 79.1378),
        "Tiruppur":     (11.1085, 77.3411),
        "Hosur":        (12.7409, 77.8253),
        "Nagercoil":    (8.1833,  77.4119),
    }

    col_in, col_out = st.columns([1, 2])

    with col_in:
        st.markdown("#### 📍 Select Location")
        city_sel    = st.selectbox("City", sorted(df["city"].unique()), key="geo_city")
        all_locs    = sorted(df[df["city"] == city_sel]["locality"].unique())
        loc_sel     = st.selectbox("Locality", all_locs, key="geo_loc")
        use_manual  = st.checkbox("Override with manual coordinates")
        if use_manual:
            mlat = st.number_input("Latitude",
                                   value=CITY_CENTERS[city_sel][0],
                                   format="%.5f")
            mlon = st.number_input("Longitude",
                                   value=CITY_CENTERS[city_sel][1],
                                   format="%.5f")
        poi_type = st.multiselect(
            "Find Nearby",
            ["🏫 Schools", "🏥 Hospitals",
             "🏦 Banks", "🛒 Supermarkets"],
            default=["🏫 Schools", "🏥 Hospitals"])
        radius   = st.slider("Search Radius (km)", 1, 10, 3)
        analyze  = st.button("🔍 Analyse Location")

    with col_out:
        if analyze:
            # ── Get coordinates ───────────────────────────────
            if use_manual:
                lat, lon = mlat, mlon
            else:
                row = df[df["locality"] == loc_sel]
                if len(row) == 0:
                    st.error("No coordinates found.")
                    st.stop()
                lat = row["latitude"].mean()
                lon = row["longitude"].mean()

            cc  = CITY_CENTERS[city_sel]
            dc  = haversine(lat, lon, cc[0], cc[1])
            sub = df[df["locality"] == loc_sel]

            # ── Fetch POIs from OpenStreetMap ─────────────────
            TYPE_MAP = {
                "🏫 Schools":      ("amenity", "school",      "#1a73e8", "🏫"),
                "🏥 Hospitals":    ("amenity", "hospital",    "#c62828", "🏥"),
                "🏦 Banks":        ("amenity", "bank",        "#f57f17", "🏦"),
                "🛒 Supermarkets": ("shop",    "supermarket", "#2e7d32", "🛒"),
            }

            all_pois = []
            with st.spinner("Fetching real locations from OpenStreetMap…"):
                for ptype in poi_type:
                    if ptype not in TYPE_MAP:
                        continue
                    tag_key, tag_val, color, icon = TYPE_MAP[ptype]
                    rad_m = radius * 1000
                    query = f"""
                    [out:json][timeout:25];
                    (
                      node["{tag_key}"="{tag_val}"](around:{rad_m},{lat},{lon});
                      way["{tag_key}"="{tag_val}"](around:{rad_m},{lat},{lon});
                    );
                    out center;
                    """
                    try:
                        resp = req.post(
                            "https://overpass-api.de/api/interpreter",
                            data=query, timeout=20)
                        data = resp.json()
                        for el in data.get("elements", []):
                            elat = (el.get("lat") or
                                    el.get("center", {}).get("lat"))
                            elon = (el.get("lon") or
                                    el.get("center", {}).get("lon"))
                            name = el.get("tags", {}).get("name", "Unnamed")
                            if elat and elon and name != "Unnamed":
                                all_pois.append({
                                    "name":  name,
                                    "type":  ptype,
                                    "lat":   elat,
                                    "lon":   elon,
                                    "color": color,
                                    "icon":  icon,
                                })
                    except Exception:
                        st.warning(f"Could not fetch {ptype} — try again.")

            # ── Scores ────────────────────────────────────────
            school_cnt = sum(1 for p in all_pois if "School" in p["type"])
            hosp_cnt   = sum(1 for p in all_pois if "Hospital" in p["type"])
            bank_cnt   = sum(1 for p in all_pois if "Bank" in p["type"])
            shop_cnt   = sum(1 for p in all_pois if "Super" in p["type"])

            s_center   = round(max(0, min(10, 10 - dc * 0.6)), 1)
            s_school   = round(min(10, school_cnt * 1.2), 1)
            s_hosp     = round(min(10, hosp_cnt   * 1.8), 1)
            s_bank     = round(min(10, bank_cnt   * 1.0), 1)
            s_shop     = round(min(10, shop_cnt   * 1.5), 1)
            s_amenity  = round(sub["amenities_score"].mean(), 1) if len(sub) > 0 else 5.0
            overall    = round((s_center + s_school + s_hosp +
                                s_bank   + s_shop   + s_amenity) / 6, 1)
            grade      = ("A+" if overall >= 8.5 else
                          "A"  if overall >= 7.0 else
                          "B+" if overall >= 6.0 else
                          "B"  if overall >= 5.0 else "C")
            grade_color = ("#3fb950" if overall >= 7 else
                           "#e3b341" if overall >= 5 else "#f78166")

            # ── Header result card ────────────────────────────
            loc_avg  = sub["price_inr"].mean() if len(sub) > 0 else 0
            city_avg = df[df["city"] == city_sel]["price_inr"].mean()
            vs_city  = ((loc_avg - city_avg) / city_avg * 100
                        if city_avg > 0 else 0)

            st.markdown(f"""
            <div class="res-card">
              <div style="display:flex;justify-content:space-between;
                          align-items:center;flex-wrap:wrap;gap:12px">
                <div>
                  <div style="font-size:1.2rem;font-weight:800;
                              color:#1a1a1a">
                    📍 {loc_sel.split(",")[0]}
                  </div>
                  <div style="font-size:.82rem;color:#555;margin-top:3px">
                    {city_sel} · {len(all_pois)} facilities found
                    within {radius} km
                  </div>
                </div>
                <div style="text-align:center">
                  <div style="font-size:3rem;font-weight:900;
                              color:{grade_color}">{grade}</div>
                  <div style="font-size:.72rem;color:#555">
                    Accessibility Grade
                  </div>
                </div>
                <div style="text-align:center">
                  <div style="font-size:2.2rem;font-weight:800;
                              color:#1a73e8">{overall}/10</div>
                  <div style="font-size:.72rem;color:#555">
                    Overall Score
                  </div>
                </div>
              </div>
            </div>""", unsafe_allow_html=True)

            # ── KPI row ───────────────────────────────────────
            st.markdown("<br>", unsafe_allow_html=True)
            c1, c2, c3, c4, c5 = st.columns(5)
            c1.markdown(kpi_card("🏙️ City Centre",
                                  f"{dc:.1f} km"),
                        unsafe_allow_html=True)
            c2.markdown(kpi_card("🏫 Schools",
                                  str(school_cnt),
                                  f"Within {radius} km"),
                        unsafe_allow_html=True)
            c3.markdown(kpi_card("🏥 Hospitals",
                                  str(hosp_cnt),
                                  f"Within {radius} km"),
                        unsafe_allow_html=True)
            c4.markdown(kpi_card("🏦 Banks",
                                  str(bank_cnt),
                                  f"Within {radius} km"),
                        unsafe_allow_html=True)
            c5.markdown(kpi_card("🛒 Supermarkets",
                                  str(shop_cnt),
                                  f"Within {radius} km"),
                        unsafe_allow_html=True)

            st.markdown("<br>", unsafe_allow_html=True)

            # ── Score bars ────────────────────────────────────
            st.markdown("#### 📊 Location Score Breakdown")
            score_bar("🏙️ City Centre Proximity",
                      s_center,  color="#1a73e8")
            score_bar("🏫 School Accessibility",
                      s_school,  color="#3fb950")
            score_bar("🏥 Hospital Proximity",
                      s_hosp,    color="#c62828")
            score_bar("🏦 Banking Access",
                      s_bank,    color="#f57f17")
            score_bar("🛒 Shopping Access",
                      s_shop,    color="#2e7d32")
            score_bar("🏊 Amenities Quality",
                      s_amenity, color="#a371f7")

            # ── Price intelligence ────────────────────────────
            if len(sub) > 0:
                st.markdown(f"""
                <div class="res-card" style="margin-top:12px">
                  <b>💰 Price Intelligence</b>
                  <div style="display:grid;
                              grid-template-columns:1fr 1fr;
                              gap:16px;margin-top:12px">
                    <div style="text-align:center">
                      <div class="kpi-lbl">
                        Avg Price in {loc_sel.split(",")[0]}
                      </div>
                      <div style="font-size:1.4rem;font-weight:700;
                                  color:#1a73e8">
                        {fmt_inr(int(loc_avg))}
                      </div>
                    </div>
                    <div style="text-align:center">
                      <div class="kpi-lbl">
                        vs {city_sel} Average
                      </div>
                      <div style="font-size:1.4rem;font-weight:700;
                                  color:{'#3fb950' if vs_city<0 else '#f78166'}">
                        {'+' if vs_city>0 else ''}{vs_city:.1f}%
                      </div>
                    </div>
                  </div>
                </div>""", unsafe_allow_html=True)

            # ── Map ───────────────────────────────────────────
            st.markdown("#### 🗺️ Location Map with Real POIs")
            from folium.plugins import Fullscreen
            m = folium.Map(location=[lat, lon],
                           zoom_start=14,
                           tiles="CartoDB positron")
            Fullscreen(position="topright").add_to(m)

            # home marker
            folium.Marker(
                [lat, lon],
                popup=f"📍 {loc_sel}",
                icon=folium.Icon(color="blue",
                                  icon="home", prefix="fa")
            ).add_to(m)

            # city centre marker
            folium.Marker(
                cc,
                popup=f"🏙️ {city_sel} Centre",
                icon=folium.Icon(color="red",
                                  icon="star", prefix="fa")
            ).add_to(m)

            # search radius circle
            folium.Circle(
                [lat, lon],
                radius=radius * 1000,
                color="#1a73e8",
                fill=True,
                fill_opacity=0.04,
                weight=2,
                dash_array="6"
            ).add_to(m)

            # nearby properties
            nearby = df[
                (abs(df["latitude"]  - lat) < 0.05) &
                (abs(df["longitude"] - lon) < 0.05)
            ].head(50)
            for _, r in nearby.iterrows():
                folium.CircleMarker(
                    [r["latitude"], r["longitude"]],
                    radius=4,
                    color="#888888",
                    fill=True,
                    fill_opacity=0.4,
                    popup=f"{r['locality']}<br>{fmt_inr(r['price_inr'])}"
                ).add_to(m)

            # POI markers
            for poi in all_pois:
                folium.CircleMarker(
                    [poi["lat"], poi["lon"]],
                    radius=8,
                    color=poi["color"],
                    fill=True,
                    fill_color=poi["color"],
                    fill_opacity=0.85,
                    weight=2,
                    popup=folium.Popup(
                        f"<b>{poi['icon']} {poi['name']}</b><br>"
                        f"<span style='color:#555'>"
                        f"{poi['type']}</span>",
                        max_width=200)
                ).add_to(m)

            # legend
            legend_html = """
            <div style='position:fixed;bottom:20px;left:20px;
                        z-index:9999;background:white;
                        border:2px solid #ccc;border-radius:10px;
                        padding:10px 14px;font-size:12px;
                        box-shadow:0 4px 12px rgba(0,0,0,0.15)'>
              <b style='color:#1a1a1a'>Legend</b><br>
              <span style='color:#1a73e8;font-size:16px'>●</span>
                Schools<br>
              <span style='color:#c62828;font-size:16px'>●</span>
                Hospitals<br>
              <span style='color:#f57f17;font-size:16px'>●</span>
                Banks<br>
              <span style='color:#2e7d32;font-size:16px'>●</span>
                Supermarkets<br>
              <span style='color:#888888;font-size:16px'>●</span>
                Nearby Properties<br>
              <span style='color:#1a73e8'>🏠</span>
                Selected Location
            </div>"""
            m.get_root().html.add_child(folium.Element(legend_html))
            st_folium(m, width=700, height=480,
                      returned_objects=[])

            # ── POI Table ─────────────────────────────────────
            if len(all_pois) > 0:
                st.markdown("#### 📋 All Nearby Facilities")
                poi_df = pd.DataFrame(all_pois)[
                    ["icon", "name", "type"]]
                poi_df.columns = ["", "Name", "Category"]
                poi_df = poi_df.sort_values("Category")
                st.dataframe(poi_df,
                             use_container_width=True,
                             hide_index=True)
            else:
                st.info("No named facilities found in this area."
                        " Try increasing the search radius.")

        else:
            st.markdown("""
            <div class="res-card" style="text-align:center;
                                         padding:80px">
              <div style="font-size:3rem">🗺️</div>
              <div style="font-size:1.15rem;font-weight:600;
                          color:#1a73e8;margin:14px 0">
                Geo Intelligence
              </div>
              <div style="font-size:.85rem;color:#8b949e">
                Select a city and locality, choose what<br>
                to find nearby, then click Analyse Location
              </div>
            </div>""", unsafe_allow_html=True)

# ══════════════════════════════════════════════════════════════
#  PAGE 3 — PRICE PREDICTION
# ══════════════════════════════════════════════════════════════
elif page == "🏡 Price Prediction":
    sec_header("🏡", "Smart Price Prediction", "XGBoost AI valuation with SHAP explainability")

    col_f, col_o = st.columns([1, 1.45])
    with col_f:
        st.markdown("#### 🔧 Property Details")
        city      = st.selectbox("City", sorted(df["city"].unique()))
        locs_city = sorted(df[df["city"]==city]["locality"].unique())
        locality  = st.selectbox("Locality", locs_city)
        prop_type = st.selectbox("Property Type", df["property_type"].unique())
        area      = st.slider("Area (sqft)", 400, 5000, 1200, 50)
        bedrooms  = st.selectbox("Bedrooms (BHK)", [1,2,3,4,5], index=1)
        bathrooms = st.selectbox("Bathrooms", [1,2,3,4,5], index=1)
        floor     = st.slider("Floor", 0, 25, 3)
        total_flr = st.slider("Total Floors in Building", max(1,floor), 30, max(floor+4,10))
        age       = st.slider("Property Age (years)", 0, 30, 4)
        parking   = st.selectbox("Parking Slots", [0,1,2])
        furnished = st.selectbox("Furnishing Status", ["Unfurnished","Semi-Furnished","Furnished"])
        facing    = st.selectbox("Facing Direction", ["East","North-East","North","North-West","West","South","South-West","South-East"])
        amenities = st.slider("Amenities Score (0-10)", 0, 10, 6)
        st.markdown("#### 📍 Distances")
        dm = st.slider("Distance to Metro/Rail (km)", 0.1, 8.0, 1.5, 0.1)
        ds = st.slider("Distance to School (km)", 0.1, 5.0, 0.8, 0.1)
        dh = st.slider("Distance to Hospital (km)", 0.1, 5.0, 1.2, 0.1)
        dw = st.slider("Distance to Highway (km)", 0.1, 8.0, 2.0, 0.1)
        predict_btn = st.button("🚀 Predict Price")

    with col_o:
        if predict_btn:
            with st.spinner("Running AI valuation…"):
                time.sleep(1.0)
                loc_row = df[df["locality"]==locality]
                dc = loc_row["dist_city_center_km"].mean() if len(loc_row)>0 else 5.0
                city_enc  = encoders["city"].transform([city])[0]
                furn_enc  = encoders["furnished"].transform([furnished])[0]
                pt_enc    = encoders["property_type"].transform([prop_type])[0]
                fac_enc   = encoders["facing"].transform([facing])[0] if facing in encoders["facing"].classes_ else 0
                inp = np.array([[area, bedrooms, bathrooms, floor, total_flr,
                                 age, parking, amenities,
                                 furn_enc, pt_enc, fac_enc, city_enc,
                                 dc, ds, dh, dm, dw]])
                pred = model.predict(inp)[0]
                low  = pred * 0.91
                high = pred * 1.09
                ppsf = pred / area
                conf = round(min(96, max(72, r2v * 100 - abs(np.random.normal(0,1.5)))), 1)

                loc_avg = df[df["locality"]==locality]["price_inr"].mean() if len(loc_row)>2 else pred
                inv_rat = "⭐⭐⭐⭐⭐" if pred < loc_avg*0.88 else \
                          "⭐⭐⭐⭐"   if pred < loc_avg*0.97 else \
                          "⭐⭐⭐"     if pred < loc_avg*1.08 else "⭐⭐"
                shap_vals = explainer.shap_values(inp)[0]

            st.markdown(f"""
            <div class="res-card">
              <div class="kpi-lbl">Estimated Market Value — {locality}</div>
              <div class="price-big">{fmt_inr(pred)}</div>
              <div style="margin-top:8px">
                <span class="badge b-blue">Low: {fmt_inr(low)}</span>
                <span class="badge b-green">High: {fmt_inr(high)}</span>
                <span class="badge b-purple">{prop_type}</span>
              </div>
            </div>""", unsafe_allow_html=True)

            c1,c2,c3,c4 = st.columns(4)
            c1.markdown(kpi_card("Price/sqft", f"₹{ppsf:,.0f}"), unsafe_allow_html=True)
            c2.markdown(kpi_card("Confidence", f"{conf}%", color="var(--green)"), unsafe_allow_html=True)
            c3.markdown(kpi_card("Investment", inv_rat), unsafe_allow_html=True)
            est_rent = int(pred * 0.003)
            c4.markdown(kpi_card("Est. Monthly Rent", fmt_inr(est_rent)), unsafe_allow_html=True)

            # Gauge
            fig_g = go.Figure(go.Indicator(
                mode="gauge+number",
                value=conf,
                number={"suffix":"%","font":{"color":"#58a6ff","size":32}},
                gauge=dict(
                    axis=dict(range=[0,100],tickcolor="#8b949e"),
                    bar=dict(color="#58a6ff"),
                    bgcolor="#21262d",
                    steps=[dict(range=[0,50],color="#30363d"),
                           dict(range=[50,75],color="#21262d"),
                           dict(range=[75,100],color="#161b22")],
                    threshold=dict(line=dict(color="#3fb950",width=3),value=conf)
                ),
                title={"text":"Prediction Confidence","font":{"color":"#8b949e","size":13}}
            ))
            fig_g.update_layout(paper_bgcolor="#1c2128",font_color="#e6edf3",
                                 height=210,margin=dict(l=30,r=30,t=40,b=10))
            st.plotly_chart(fig_g, use_container_width=True)

            # SHAP waterfall
            shap_df = pd.DataFrame({"Feature":feat_cols,"SHAP":shap_vals})\
                        .sort_values("SHAP",key=abs,ascending=True).tail(10)
            colors = ["#f78166" if v<0 else "#58a6ff" for v in shap_df["SHAP"]]
            fig_sh = go.Figure(go.Bar(
                x=shap_df["SHAP"], y=shap_df["Feature"], orientation="h",
                marker_color=colors
            ))
            fig_sh.update_layout(**PLOTLY_LAYOUT, height=280,
                xaxis=dict(gridcolor="#30363d",color="#8b949e",title="SHAP Contribution (₹)"),
                yaxis=dict(color="#8b949e"))
            st.plotly_chart(fig_sh, use_container_width=True)

            # What-if analysis
            st.markdown("#### 🔄 What-If Analysis — Change Area")
            areas_wif = list(range(500, 4001, 100))
            preds_wif = []
            for a in areas_wif:
                inp2 = inp.copy(); inp2[0][0] = a
                preds_wif.append(model.predict(inp2)[0])
            fig_wif = px.line(x=areas_wif, y=preds_wif,
                              labels={"x":"Area (sqft)","y":"Predicted Price (₹)"},
                              color_discrete_sequence=["#3fb950"])
            fig_wif.add_vline(x=area, line_dash="dash", line_color="#58a6ff",
                              annotation_text=f"Current: {area} sqft")
            fig_wif.update_layout(**PLOTLY_LAYOUT,
                xaxis=dict(gridcolor="#30363d",color="#8b949e"),
                yaxis=dict(gridcolor="#30363d",color="#8b949e"))
            st.plotly_chart(fig_wif, use_container_width=True)

        else:
            st.markdown("""<div class="res-card" style="text-align:center;padding:80px">
              <div style="font-size:3rem">🤖</div>
              <div style="font-size:1.15rem;font-weight:600;color:#58a6ff;margin:14px 0">
                Tamil Nadu AI Valuation Engine
              </div>
              <div style="font-size:.85rem;color:#8b949e">
                Fill property details on the left<br>and click Predict Price
              </div></div>""", unsafe_allow_html=True)


# ══════════════════════════════════════════════════════════════
#  PAGE 4 — IMAGE INTELLIGENCE
# ══════════════════════════════════════════════════════════════
elif page == "🖼️ Image Intelligence":
    sec_header("🖼️", "Image Intelligence", "Upload multiple room photos — AI predicts property quality & price range")

    st.markdown("""
    <div class="res-card" style="margin-bottom:18px">
      <b style="color:#1a73e8">📸 How it works:</b>
      <div style="font-size:.85rem;color:#555555;margin-top:6px;line-height:1.9">
        Upload photos of each room → AI analyses brightness, contrast, texture, luxury score per room
        → Combines all room scores → Predicts overall property grade + estimated price range
      </div>
    </div>
    """, unsafe_allow_html=True)

    # ── Room upload slots ─────────────────────────────────────
    ROOMS = [
        ("🏠 Exterior / Front View",   "exterior"),
        ("🛋️ Living Room",             "living"),
        ("🛏️ Master Bedroom",          "bedroom"),
        ("🍳 Kitchen",                 "kitchen"),
        ("🚿 Bathroom",                "bathroom"),
        ("🏡 Other Room / Balcony",    "other"),
    ]

    st.markdown("#### 📤 Upload Room Photos (upload as many as you have)")
    cols_up = st.columns(3)
    uploaded_rooms = {}
    for i, (room_label, room_key) in enumerate(ROOMS):
        with cols_up[i % 3]:
            f = st.file_uploader(room_label, type=["jpg","jpeg","png"], key=f"img_{room_key}")
            if f:
                uploaded_rooms[room_label] = f

    # ── City + Property type for price estimation ─────────────
    st.markdown("#### 📍 Property Info for Price Estimate")
    ci1, ci2, ci3 = st.columns(3)
    with ci1:
        img_city = st.selectbox("City", sorted(df["city"].unique()), key="img_city")
    with ci2:
        img_type = st.selectbox("Property Type", df["property_type"].unique(), key="img_type")
    with ci3:
        img_area = st.slider("Approx Area (sqft)", 400, 5000, 1200, 50, key="img_area")

    analyze_btn = st.button("🔬 Analyse All Uploaded Rooms", disabled=len(uploaded_rooms)==0)

    if len(uploaded_rooms) == 0:
        st.markdown("""<div class="res-card" style="text-align:center;padding:60px">
          <div style="font-size:3rem">🖼️</div>
          <div style="font-size:1rem;color:#8b949e;margin-top:12px">
            Upload at least one room photo above to begin analysis
          </div></div>""", unsafe_allow_html=True)

    elif analyze_btn:
        with st.spinner("Analysing all room images with AI…"):
            time.sleep(1.0)

            def analyse_image(img_file):
                img  = Image.open(img_file).convert("RGB")
                arr  = np.array(img.resize((224,224)), dtype=np.float32) / 255.0
                brightness  = round(float(arr.mean() * 10), 1)
                contrast    = round(float(min(10, arr.std() * 32)), 1)
                r,g,b       = arr[:,:,0].mean(), arr[:,:,1].mean(), arr[:,:,2].mean()
                texture     = round(float(min(10, np.std(np.diff(arr[:,:,0], axis=0))*85+2.5)), 1)
                color_harm  = round(float(max(2.5, min(10, 10-abs(r-g)*5-abs(g-b)*3))), 1)
                space_sc    = round(float(min(10, max(2, 4.5+brightness*0.4+np.random.uniform(-.3,.3)))), 1)
                luxury_p    = round(float(min(.97, (brightness/10+texture/10+color_harm/10)/3+np.random.uniform(-.04,.09))), 2)
                overall     = round(float(max(3, min(10, (brightness+contrast+texture+color_harm+space_sc)/5))), 1)
                return {
                    "brightness": brightness, "contrast": contrast,
                    "texture": texture, "color_harm": color_harm,
                    "space": space_sc, "luxury": luxury_p, "overall": overall,
                    "img": img
                }

            room_results = {}
            for room_label, img_file in uploaded_rooms.items():
                room_results[room_label] = analyse_image(img_file)

        # ── Show each room result ─────────────────────────────
        st.markdown("---")
        st.markdown("### 🔍 Room-by-Room Analysis")

        room_scores = []
        for room_label, res in room_results.items():
            room_scores.append(res["overall"])
            with st.expander(f"{room_label}  —  Score: {res['overall']}/10", expanded=True):
                c_img, c_scores = st.columns([1, 1.6])
                with c_img:
                    st.image(res["img"], use_container_width=True, caption=room_label)
                with c_scores:
                    grade = "A+" if res["overall"]>=8.5 else "A" if res["overall"]>=7.5 else "B+" if res["overall"]>=6.5 else "B" if res["overall"]>=5.5 else "C"
                    grade_color = "#3fb950" if res["overall"]>=7 else "#e3b341" if res["overall"]>=5 else "#f78166"
                    st.markdown(f"""
                    <div style="display:flex;gap:20px;align-items:center;margin-bottom:12px">
                      <div style="font-size:2.8rem;font-weight:900;color:{grade_color}">{grade}</div>
                      <div>
                        <div style="font-size:1.6rem;font-weight:800;color:#1a73e8">{res['overall']}/10</div>
                        <div style="font-size:.75rem;color:#555">Overall Room Score</div>
                      </div>
                      <div>
                        <span class="badge {'b-green' if res['luxury']>=0.65 else 'b-gold'}">
                          Luxury: {res['luxury']*100:.0f}%
                        </span>
                      </div>
                    </div>
                    """, unsafe_allow_html=True)
                    score_bar("☀️ Brightness & Lighting", res["brightness"])
                    score_bar("🎨 Contrast & Clarity",    res["contrast"],   color="#3fb950")
                    score_bar("🧱 Texture Richness",       res["texture"],    color="#e3b341")
                    score_bar("🌈 Color Harmony",          res["color_harm"], color="#f78166")
                    score_bar("📐 Space Perception",       res["space"],      color="#a371f7")

        # ── Overall property summary ──────────────────────────
        st.markdown("---")
        st.markdown("### 🏠 Overall Property Assessment")

        overall_score = round(sum(room_scores) / len(room_scores), 1)
        overall_grade = ("A+" if overall_score>=8.5 else "A" if overall_score>=7.5 else
                         "B+" if overall_score>=6.5 else "B" if overall_score>=5.5 else "C")
        overall_color = "#3fb950" if overall_score>=7 else "#e3b341" if overall_score>=5 else "#f78166"

        # Price estimation based on image quality score
        city_data    = df[df["city"] == img_city]
        type_data    = city_data[city_data["property_type"] == img_type]
        base_data    = type_data if len(type_data) > 5 else city_data
        base_psf     = (base_data["price_inr"] / base_data["area_sqft"]).median()
        quality_mult = 0.75 + (overall_score / 10) * 0.55   # 0.75x to 1.30x based on quality
        est_psf      = base_psf * quality_mult
        est_price    = est_psf * img_area
        est_low      = est_price * 0.90
        est_high     = est_price * 1.10

        cs1, cs2, cs3 = st.columns(3)
        cs1.markdown(kpi_card("Overall Score",   f"{overall_score}/10",  color=overall_color), unsafe_allow_html=True)
        cs2.markdown(kpi_card("Property Grade",  overall_grade,          color=overall_color), unsafe_allow_html=True)
        cs3.markdown(kpi_card("Rooms Analysed",  str(len(room_results))), unsafe_allow_html=True)

        st.markdown("<br>", unsafe_allow_html=True)

        # Price estimate card
        st.markdown(f"""
        <div class="res-card" style="text-align:center;padding:28px">
          <div class="kpi-lbl">AI Estimated Price Range — {img_city} · {img_type} · {img_area} sqft</div>
          <div class="price-big">{fmt_inr(int(est_price))}</div>
          <div style="margin-top:10px">
            <span class="badge b-blue">Low: {fmt_inr(int(est_low))}</span>
            <span class="badge b-green">High: {fmt_inr(int(est_high))}</span>
            <span class="badge b-purple">₹{int(est_psf):,}/sqft</span>
          </div>
          <div style="font-size:.78rem;color:#888;margin-top:10px">
            Price adjusted by image quality score ({overall_score}/10) against {img_city} {img_type} market average
          </div>
        </div>
        """, unsafe_allow_html=True)

        st.markdown("<br>", unsafe_allow_html=True)

        # Room score comparison bar chart
        st.markdown("#### 📊 Room Score Comparison")
        fig_rooms = go.Figure(go.Bar(
            x=list(room_results.keys()),
            y=[r["overall"] for r in room_results.values()],
            marker_color=["#3fb950" if r["overall"]>=7 else "#e3b341" if r["overall"]>=5
                          else "#f78166" for r in room_results.values()],
            text=[f"{r['overall']}/10" for r in room_results.values()],
            textposition="outside"
        ))
        fig_rooms.add_hline(y=overall_score, line_dash="dash", line_color="#1a73e8",
                            annotation_text=f"Avg: {overall_score}", annotation_position="right")
        fig_rooms.update_layout(**PLOTLY_LAYOUT, height=320,
            xaxis=dict(color="#333333", tickangle=-15),
            yaxis=dict(gridcolor="#eeeeee", color="#333333", range=[0,11], title="Score /10"))
        st.plotly_chart(fig_rooms, use_container_width=True)

        # Radar — average across all rooms
        cats  = ["Brightness","Contrast","Texture","Color","Space"]
        avgs  = [
            round(sum(r["brightness"] for r in room_results.values())/len(room_results),1),
            round(sum(r["contrast"]   for r in room_results.values())/len(room_results),1),
            round(sum(r["texture"]    for r in room_results.values())/len(room_results),1),
            round(sum(r["color_harm"] for r in room_results.values())/len(room_results),1),
            round(sum(r["space"]      for r in room_results.values())/len(room_results),1),
        ]
        fig_rad = go.Figure(go.Scatterpolar(
            r=avgs+[avgs[0]], theta=cats+[cats[0]],
            fill="toself", fillcolor="rgba(26,115,232,.12)", line_color="#1a73e8"
        ))
        fig_rad.update_layout(
            polar=dict(
                radialaxis=dict(visible=True, range=[0,10], color="#333333", gridcolor="#eeeeee"),
                angularaxis=dict(color="#333333"),
                bgcolor="#f8f9fa"
            ),
            paper_bgcolor="#ffffff", font_color="#1a1a1a",
            showlegend=False, height=350,
            margin=dict(l=40,r=40,t=30,b=20)
        )
        st.plotly_chart(fig_rad, use_container_width=True)

        # Verdict
        verdict = ("🏆 Excellent property — premium finishes, great lighting and space across all rooms."
                   if overall_score >= 8 else
                   "✅ Good quality property — well maintained with decent interiors."
                   if overall_score >= 6.5 else
                   "🔶 Average property — some rooms need improvement."
                   if overall_score >= 5 else
                   "⚠️ Below average — significant renovation recommended before listing.")
        reno = ("No renovation needed" if overall_score>=8 else
                "Minor touch-ups"      if overall_score>=6.5 else
                "Moderate renovation"  if overall_score>=5 else
                "Full renovation recommended")
        weak_room = min(room_results, key=lambda x: room_results[x]["overall"])

        st.markdown(f"""
        <div class="res-card" style="background:#e8f5e9;border-color:#2e7d32">
          <b style="color:#2e7d32">🤖 AI Property Verdict</b>
          <div style="margin-top:8px;font-size:.92rem;color:#1a1a1a">{verdict}</div>
          <div style="display:grid;grid-template-columns:1fr 1fr 1fr;gap:12px;margin-top:14px">
            <div style="text-align:center">
              <div class="kpi-lbl">Renovation Needed</div>
              <div style="font-weight:700;color:#1a73e8;font-size:.9rem">{reno}</div>
            </div>
            <div style="text-align:center">
              <div class="kpi-lbl">Weakest Room</div>
              <div style="font-weight:700;color:#f78166;font-size:.9rem">{weak_room}</div>
            </div>
            <div style="text-align:center">
              <div class="kpi-lbl">Best Room</div>
              <div style="font-weight:700;color:#3fb950;font-size:.9rem">{max(room_results, key=lambda x: room_results[x]["overall"])}</div>
            </div>
          </div>
        </div>
        """, unsafe_allow_html=True)
# ══════════════════════════════════════════════════════════════
#  PAGE 6 — MODEL EXPLAINABILITY
# ══════════════════════════════════════════════════════════════
elif page == "🧠 Model Explainability":
    sec_header("🧠", "Model Explainability", "SHAP-based Explainable AI — why prices are what they are")

    c1,c2 = st.columns([1,2])
    with c1:
        st.markdown("#### 📐 Model Metrics")
        st.markdown(kpi_card("Algorithm", "XGBoost", "Gradient Boosting"), unsafe_allow_html=True)
        st.markdown("<br>", unsafe_allow_html=True)
        st.markdown(kpi_card("R² Score", f"{r2v:.4f}", "Variance explained", color="var(--green)"), unsafe_allow_html=True)
        st.markdown("<br>", unsafe_allow_html=True)
        st.markdown(kpi_card("MAE", fmt_inr(mae), "Mean absolute error", color="var(--gold)"), unsafe_allow_html=True)
        st.markdown("<br>", unsafe_allow_html=True)
        st.markdown(kpi_card("Features", str(len(feat_cols)), "Input variables"), unsafe_allow_html=True)
        st.markdown("<br>", unsafe_allow_html=True)
        st.markdown(kpi_card("Training Data", f"{int(len(df)*.8):,}", "Properties used"), unsafe_allow_html=True)

    with c2:
        st.markdown("#### Global SHAP Feature Importance")
        sample_shap = X_test.sample(min(150, len(X_test)), random_state=42)
        shap_all    = explainer.shap_values(sample_shap)
        mean_shap   = np.abs(shap_all).mean(axis=0)
        shap_df     = pd.DataFrame({"Feature":feat_cols,"Mean |SHAP|":mean_shap})\
                        .sort_values("Mean |SHAP|",ascending=True)
        fig_shap = px.bar(shap_df, x="Mean |SHAP|", y="Feature", orientation="h",
                          color="Mean |SHAP|", color_continuous_scale="Blues",
                          labels={"Mean |SHAP|":"Mean |SHAP| Value"})
        fig_shap.update_layout(**PLOTLY_LAYOUT, coloraxis_showscale=False,
            xaxis=dict(gridcolor="#30363d",color="#8b949e"),
            yaxis=dict(color="#8b949e",tickfont=dict(size=10)))
        st.plotly_chart(fig_shap, use_container_width=True)

    st.markdown("---")
    st.markdown("#### 🔍 Individual Property Explanation")
    idx = st.slider("Sample property index", 0, min(99,len(X_test)-1), 0)
    s_row  = X_test.iloc[[idx]]
    s_pred = model.predict(s_row)[0]
    s_shap = explainer.shap_values(s_row)[0]
    base_v = explainer.expected_value

    st.markdown(f"""
    <div class="res-card">
      <div class="kpi-lbl">Predicted Price for Sample #{idx}</div>
      <div class="price-big">{fmt_inr(s_pred)}</div>
      <div style="font-size:.82rem;color:#8b949e;margin-top:6px">
        Base (average) value: {fmt_inr(base_v)} &nbsp;→&nbsp; After feature adjustments: {fmt_inr(s_pred)}
      </div>
    </div>""", unsafe_allow_html=True)

    contrib = pd.DataFrame({"Feature":feat_cols,"SHAP":s_shap})\
                .sort_values("SHAP",key=abs,ascending=False)
    colors2 = ["#58a6ff" if v>=0 else "#f78166" for v in contrib["SHAP"]]
    fig_c = go.Figure(go.Bar(
        x=contrib["Feature"], y=contrib["SHAP"], marker_color=colors2,
        text=[f"{'+' if v>0 else ''}{fmt_inr(v)}" for v in contrib["SHAP"]],
        textposition="outside"
    ))
    fig_c.update_layout(**PLOTLY_LAYOUT, height=360,
        xaxis=dict(color="#8b949e",tickangle=-30),
        yaxis=dict(gridcolor="#30363d",color="#8b949e",title="SHAP Value (₹)"))
    st.plotly_chart(fig_c, use_container_width=True)

    # Natural language explanation
    top3 = contrib.head(3)
    bullets = ""
    for _, r in top3.iterrows():
        sign  = "increased" if r["SHAP"]>0 else "decreased"
        color = "#3fb950"   if r["SHAP"]>0 else "#f78166"
        bullets += f'<li><span style="color:{color};font-weight:600">{r["Feature"]}</span> {sign} price by <b>{fmt_inr(abs(r["SHAP"]))}</b></li>'
    st.markdown(f"""
    <div class="res-card">
      <b>🗣️ AI Explanation in Plain Language</b>
      <ul style="margin-top:10px;color:#8b949e;font-size:.87rem;line-height:2">{bullets}</ul>
    </div>""", unsafe_allow_html=True)

    # Actual vs Predicted
    st.markdown("#### Actual vs Predicted Prices (Test Set)")
    act = y_test.values[:150]
    prd = model.predict(X_test[:150])
    fig_avp = go.Figure()
    fig_avp.add_trace(go.Scatter(x=act, y=prd, mode="markers",
                                  marker=dict(color="#58a6ff",opacity=.6,size=6),
                                  name="Predictions"))
    mn,mx = min(act.min(),prd.min()), max(act.max(),prd.max())
    fig_avp.add_trace(go.Scatter(x=[mn,mx],y=[mn,mx],mode="lines",
                                  line=dict(color="#3fb950",dash="dash"),name="Perfect Fit"))
    fig_avp.update_layout(**PLOTLY_LAYOUT,
        xaxis=dict(gridcolor="#30363d",color="#8b949e",title="Actual Price (₹)"),
        yaxis=dict(gridcolor="#30363d",color="#8b949e",title="Predicted Price (₹)"),
        legend=dict(bgcolor="#1c2128",bordercolor="#30363d"))
    st.plotly_chart(fig_avp, use_container_width=True)


# ══════════════════════════════════════════════════════════════
#  PAGE 7 — INVESTMENT ADVISOR
# ══════════════════════════════════════════════════════════════
elif page == "📈 Investment Advisor":
    sec_header("📈", "Investment Advisor", "AI-powered Tamil Nadu investment recommendations")

    col_f, col_r = st.columns([1,1.6])
    with col_f:
        st.markdown("#### 💰 Your Profile")
        budget    = st.number_input("Budget (₹ in Lakhs)", 20, 5000, 80, 5) * 100_000
        purpose   = st.selectbox("Purpose", ["End Use","Rental Income","Capital Appreciation","All"])
        city_pref = st.multiselect("Preferred Cities", sorted(df["city"].unique()),
                                   default=["Chennai","Coimbatore"])
        bhk_pref  = st.multiselect("Preferred BHK", [1,2,3,4,5], default=[2,3])
        prop_pref = st.multiselect("Property Type", df["property_type"].unique(),
                                   default=["Apartment"])
        horizon   = st.selectbox("Investment Horizon", ["Short (1-2 yrs)","Medium (3-5 yrs)","Long (7+ yrs)"])
        risk      = st.radio("Risk Appetite", ["Conservative","Moderate","Aggressive"])
        go_btn    = st.button("🎯 Get Recommendations")

    with col_r:
        if go_btn:
            with st.spinner("Analyzing Tamil Nadu market…"):
                time.sleep(1.2)
                filt = df[df["price_inr"] <= budget*1.12].copy()
                if city_pref:  filt = filt[filt["city"].isin(city_pref)]
                if bhk_pref:   filt = filt[filt["bedrooms"].isin(bhk_pref)]
                if prop_pref:  filt = filt[filt["property_type"].isin(prop_pref)]

                if len(filt) < 5:
                    st.warning("⚠️ Too few properties match your criteria. Widening search…")
                    filt = df[df["price_inr"] <= budget*1.2].copy()

                avg_psf_overall = (df["price_inr"]/df["area_sqft"]).mean()
                filt["psf"]      = filt["price_inr"]/filt["area_sqft"]
                filt["value_sc"] = (avg_psf_overall - filt["psf"])/avg_psf_overall*10
                filt["loc_sc"]   = (10 - filt["dist_metro_km"]*1.2 - filt["dist_city_center_km"]*0.5
                                    + filt["amenities_score"]*0.3).clip(0,10)
                filt["roi_sc"]   = (filt["value_sc"]*0.5 + filt["loc_sc"]*0.5).clip(0,10)
                filt["rent_est"] = filt["price_inr"]*0.003
                growth_map = {"Short (1-2 yrs)":6,"Medium (3-5 yrs)":11,"Long (7+ yrs)":14}
                filt["apprec"]   = filt["roi_sc"] * 1.3 + growth_map.get(horizon,11) * 0.5

                summary = filt.groupby(["locality","city"]).agg(
                    Avg_Price=("price_inr","mean"),
                    ROI=("roi_sc","mean"),
                    Rent=("rent_est","mean"),
                    Appreciation=("apprec","mean"),
                    Count=("price_inr","count")
                ).sort_values("ROI",ascending=False).head(6).reset_index()

            st.markdown("#### 🏆 Top Recommended Areas")
            medals = ["🥇","🥈","🥉","4️⃣","5️⃣","6️⃣"]
            for rank,(_, r) in enumerate(summary.iterrows(), 1):
                st.markdown(f"""
                <div class="res-card" style="margin-bottom:10px">
                  <div style="display:flex;justify-content:space-between;align-items:center">
                    <div>
                      <span style="font-size:1.05rem;font-weight:700">{medals[rank-1]} {r['locality']}</span>
                      <span class="badge b-blue">{r['city']}</span>
                      <span class="badge b-purple">{r['Count']:.0f} in budget</span>
                    </div>
                    <div style="font-size:1.4rem;font-weight:800;color:#3fb950">ROI {r['ROI']:.1f}/10</div>
                  </div>
                  <div style="display:grid;grid-template-columns:1fr 1fr 1fr;gap:12px;margin-top:12px">
                    <div style="text-align:center">
                      <div class="kpi-lbl">Avg Price</div>
                      <div style="font-weight:600;color:#58a6ff">{fmt_inr(r['Avg_Price'])}</div>
                    </div>
                    <div style="text-align:center">
                      <div class="kpi-lbl">Est. Monthly Rent</div>
                      <div style="font-weight:600;color:#3fb950">{fmt_inr(r['Rent'])}/mo</div>
                    </div>
                    <div style="text-align:center">
                      <div class="kpi-lbl">Annual Appreciation</div>
                      <div style="font-weight:600;color:#e3b341">{r['Appreciation']:.1f}%/yr</div>
                    </div>
                  </div>
                </div>""", unsafe_allow_html=True)

            # 5-year forecast
            st.markdown("#### 📈 5-Year Price Forecast — Top Locality")
            top_area  = summary.iloc[0]["locality"]
            base_p    = summary.iloc[0]["Avg_Price"]
            ann_g     = summary.iloc[0]["Appreciation"]/100
            years     = list(range(2024, 2030))
            prices    = [base_p*(1+ann_g)**i for i in range(len(years))]
            fig_fore  = px.line(x=years, y=prices,
                                labels={"x":"Year","y":"Projected Price (₹)"},
                                title=f"Price Forecast — {top_area}",
                                color_discrete_sequence=["#3fb950"])
            fig_fore.update_traces(mode="lines+markers",marker=dict(size=8))
            fig_fore.update_layout(**PLOTLY_LAYOUT,
                title_font_color="#58a6ff",
                xaxis=dict(gridcolor="#30363d",color="#8b949e"),
                yaxis=dict(gridcolor="#30363d",color="#8b949e"))
            st.plotly_chart(fig_fore, use_container_width=True)

            # Budget allocation pie
            st.markdown("#### 💼 Suggested Budget Allocation")
            alloc_names = ["Property Cost","Registration (7%)","Interior/Renovation","Emergency Buffer"]
            alloc_vals  = [budget*0.85, budget*0.07, budget*0.05, budget*0.03]
            fig_pie = px.pie(names=alloc_names, values=alloc_vals, hole=0.4,
                             color_discrete_sequence=["#58a6ff","#3fb950","#e3b341","#a371f7"])
            fig_pie.update_layout(**PLOTLY_LAYOUT,
                legend=dict(bgcolor="#1c2128",bordercolor="#30363d"))
            st.plotly_chart(fig_pie, use_container_width=True)

        else:
            st.markdown("""<div class="res-card" style="text-align:center;padding:80px">
              <div style="font-size:3rem">💎</div>
              <div style="font-size:1.15rem;font-weight:600;color:#58a6ff;margin:14px 0">
                Tamil Nadu Investment Engine
              </div>
              <div style="font-size:.85rem;color:#8b949e">
                Set your budget & preferences, then<br>click Get Recommendations
              </div></div>""", unsafe_allow_html=True)


# ══════════════════════════════════════════════════════════════
#  PAGE 8 — PROPERTY COMPARATOR
# ══════════════════════════════════════════════════════════════
elif page == "🔁 Property Comparator":
    sec_header("🔁", "Property Comparator", "Side-by-side AI comparison of two Tamil Nadu properties")

    def prop_inputs(label, key, def_city, def_loc_idx, def_area, def_bhk, def_age, def_furn):
        st.markdown(f"#### {label}")
        city = st.selectbox(f"City_{key}", sorted(df["city"].unique()),
                            index=list(sorted(df["city"].unique())).index(def_city) if def_city in df["city"].unique() else 0,
                            key=f"city_{key}", label_visibility="collapsed")
        locs = sorted(df[df["city"]==city]["locality"].unique())
        loc  = st.selectbox(f"Locality_{key}", locs,
                            index=min(def_loc_idx, len(locs)-1),
                            key=f"loc_{key}", label_visibility="collapsed")
        ptype= st.selectbox(f"Type_{key}", df["property_type"].unique(), key=f"pt_{key}", label_visibility="collapsed")
        area = st.slider(f"Area (sqft)_{key}", 400, 5000, def_area, 50, key=f"area_{key}")
        bhk  = st.selectbox(f"BHK_{key}", [1,2,3,4,5], index=def_bhk-1, key=f"bhk_{key}", label_visibility="collapsed")
        bat  = st.selectbox(f"Baths_{key}", [1,2,3,4,5], index=min(def_bhk-1,4), key=f"bat_{key}", label_visibility="collapsed")
        fl   = st.slider(f"Floor_{key}", 0, 20, 3, key=f"fl_{key}")
        age  = st.slider(f"Age_{key}", 0, 30, def_age, key=f"age_{key}")
        furn = st.selectbox(f"Furnish_{key}", ["Unfurnished","Semi-Furnished","Furnished"],
                            index=def_furn, key=f"furn_{key}", label_visibility="collapsed")
        amen = st.slider(f"Amenities_{key}", 0, 10, 6, key=f"amen_{key}")
        dm   = st.slider(f"Dist Metro_{key}", 0.1, 8.0, 1.5, 0.1, key=f"dm_{key}")
        return dict(city=city,loc=loc,ptype=ptype,area=area,bhk=bhk,bat=bat,
                    fl=fl,age=age,furn=furn,amen=amen,dm=dm)

    col1,_,col2 = st.columns([1,.04,1])
    with col1:
        v1 = prop_inputs("🏠 Property A","p1","Chennai",0,1200,2,5,1)
    with _:
        st.markdown("<div style='border-left:1px solid #30363d;height:100%;margin:0 auto'></div>", unsafe_allow_html=True)
    with col2:
        v2 = prop_inputs("🏡 Property B","p2","Coimbatore",2,900,2,10,0)

    if st.button("⚖️ Compare Now"):
        def predict_v(v):
            row = df[df["locality"]==v["loc"]]
            dc  = row["dist_city_center_km"].mean() if len(row)>0 else 5.0
            ds  = row["dist_school_km"].mean() if len(row)>0 else 1.0
            dh  = row["dist_hospital_km"].mean() if len(row)>0 else 1.5
            dw  = row["dist_highway_km"].mean() if len(row)>0 else 2.0
            ce  = encoders["city"].transform([v["city"]])[0]
            fe  = encoders["furnished"].transform([v["furn"]])[0]
            pe  = encoders["property_type"].transform([v["ptype"]])[0]
            fc  = encoders["facing"].transform(["East"])[0]
            inp = np.array([[v["area"],v["bhk"],v["bat"],v["fl"],v["fl"]+5,
                             v["age"],1,v["amen"],fe,pe,fc,ce,
                             dc,ds,dh,v["dm"],dw]])
            pred = model.predict(inp)[0]
            shap_v = explainer.shap_values(inp)[0]
            return pred, shap_v, inp

        p1,s1,i1 = predict_v(v1)
        p2,s2,i2 = predict_v(v2)

        st.markdown("<br>", unsafe_allow_html=True)
        ca, cb = st.columns(2)
        def winner_badge(p,o,a,oa):
            if p < o and a >= oa: return " 🏆 Better Value!"
            return ""

        with ca:
            wb = winner_badge(p1,p2,v1["area"],v2["area"])
            st.markdown(f"""<div class="res-card" style="text-align:center">
              <div class="kpi-lbl">Property A — {v1['loc'].split(',')[0]}{wb}</div>
              <div class="price-big">{fmt_inr(p1)}</div>
              <div style="color:#8b949e;font-size:.82rem">₹{p1/v1['area']:,.0f}/sqft</div>
              <span class="badge b-blue">{v1['city']}</span>
              <span class="badge b-purple">{v1['ptype']}</span>
            </div>""", unsafe_allow_html=True)
        with cb:
            wb2 = winner_badge(p2,p1,v2["area"],v1["area"])
            st.markdown(f"""<div class="res-card" style="text-align:center">
              <div class="kpi-lbl">Property B — {v2['loc'].split(',')[0]}{wb2}</div>
              <div class="price-big">{fmt_inr(p2)}</div>
              <div style="color:#8b949e;font-size:.82rem">₹{p2/v2['area']:,.0f}/sqft</div>
              <span class="badge b-green">{v2['city']}</span>
              <span class="badge b-purple">{v2['ptype']}</span>
            </div>""", unsafe_allow_html=True)

        # Radar comparison
        cats = ["Area","BHK","Bathrooms","Floor","Amenities","Value/sqft"]
        def norm(v,mn,mx): return round((v-mn)/(mx-mn)*10,1) if mx>mn else 5.0
        r1 = [norm(v1["area"],400,5000), norm(v1["bhk"],1,5), norm(v1["bat"],1,5),
              norm(v1["fl"],0,20), norm(v1["amen"],0,10), norm(1/(p1/v1["area"]),1/40000,1/2000)]
        r2 = [norm(v2["area"],400,5000), norm(v2["bhk"],1,5), norm(v2["bat"],1,5),
              norm(v2["fl"],0,20), norm(v2["amen"],0,10), norm(1/(p2/v2["area"]),1/40000,1/2000)]

        fig_rad = go.Figure()
        fig_rad.add_trace(go.Scatterpolar(r=r1+[r1[0]], theta=cats+[cats[0]],
            fill="toself", name=f"A: {v1['loc'].split(',')[0]}",
            fillcolor="rgba(88,166,255,.14)", line_color="#58a6ff"))
        fig_rad.add_trace(go.Scatterpolar(r=r2+[r2[0]], theta=cats+[cats[0]],
            fill="toself", name=f"B: {v2['loc'].split(',')[0]}",
            fillcolor="rgba(63,185,80,.14)", line_color="#3fb950"))
        fig_rad.update_layout(
            polar=dict(radialaxis=dict(visible=True,range=[0,10],color="#8b949e",gridcolor="#30363d"),
                       angularaxis=dict(color="#8b949e"),bgcolor="#1c2128"),
            paper_bgcolor="#1c2128", font_color="#e6edf3",
            legend=dict(bgcolor="#1c2128",bordercolor="#30363d"),
            height=400, margin=dict(l=40,r=40,t=30,b=20)
        )
        st.plotly_chart(fig_rad, use_container_width=True)

        # SHAP delta
        st.markdown("#### SHAP Feature Difference (A − B)")
        delta_df = pd.DataFrame({"Feature":feat_cols,"Delta":s1-s2})\
                     .sort_values("Delta",key=abs,ascending=True).tail(10)
        dcols = ["#58a6ff" if v>=0 else "#f78166" for v in delta_df["Delta"]]
        fig_d = go.Figure(go.Bar(x=delta_df["Delta"],y=delta_df["Feature"],
                                  orientation="h",marker_color=dcols))
        fig_d.update_layout(**PLOTLY_LAYOUT, height=320,
            xaxis=dict(gridcolor="#30363d",color="#8b949e",title="SHAP Δ (A−B)"),
            yaxis=dict(color="#8b949e"))
        st.plotly_chart(fig_d, use_container_width=True)
# ══════════════════════════════════════════════════════════════
#  PAGE 9 — EMI & LOAN CALCULATOR
# ══════════════════════════════════════════════════════════════
elif page == "💸 EMI Calculator":
    sec_header("💸", "EMI & Loan Calculator", "Plan your home loan with full amortization breakdown")

    col_f, col_o = st.columns([1, 1.5])
    with col_f:
        st.markdown("#### 🏦 Loan Details")
        prop_price = st.number_input("Property Price (₹ in Lakhs)", 10, 5000, 80, 5) * 100_000
        down_pct   = st.slider("Down Payment (%)", 5, 50, 20)
        down_amt   = prop_price * down_pct / 100
        loan_amt   = prop_price - down_amt
        st.info(f"💰 Loan Amount: **{fmt_inr(int(loan_amt))}**  after {down_pct}% down payment of {fmt_inr(int(down_amt))}")

        bank = st.selectbox("Bank / Lender", [
            "SBI Home Loan (8.50%)",
            "HDFC Bank (8.70%)",
            "ICICI Bank (8.75%)",
            "Axis Bank (8.75%)",
            "LIC Housing (8.65%)",
            "Bank of Baroda (8.40%)",
            "Custom Rate",
        ])
        bank_rates = {
            "SBI Home Loan (8.50%)":  8.50,
            "HDFC Bank (8.70%)":      8.70,
            "ICICI Bank (8.75%)":     8.75,
            "Axis Bank (8.75%)":      8.75,
            "LIC Housing (8.65%)":    8.65,
            "Bank of Baroda (8.40%)": 8.40,
            "Custom Rate":            None,
        }
        if bank == "Custom Rate":
            rate = st.slider("Annual Interest Rate (%)", 6.0, 15.0, 8.5, 0.05)
        else:
            rate = bank_rates[bank]
            st.markdown(f"<span class='badge b-blue'>Rate: {rate}% p.a.</span>", unsafe_allow_html=True)

        tenure_yr      = st.slider("Loan Tenure (Years)", 5, 30, 20)
        monthly_salary = st.number_input("Your Monthly Salary (₹)", 10000, 10000000, 100000, 5000)
        calc_btn       = st.button("📊 Calculate EMI")

    with col_o:
        if calc_btn:
            n   = tenure_yr * 12
            r   = rate / 100 / 12
            emi = loan_amt * r * (1 + r)**n / ((1 + r)**n - 1)
            total_pay  = emi * n
            total_int  = total_pay - loan_amt
            emi_ratio  = emi / monthly_salary * 100
            eligibility = ("✅ Eligible"       if emi_ratio <= 45 else
                           "⚠️ High Ratio"     if emi_ratio <= 60 else
                           "❌ May be Rejected")
            eli_color   = ("#3fb950" if emi_ratio <= 45 else
                           "#e3b341" if emi_ratio <= 60 else "#f78166")

            # ── KPI Row ──────────────────────────────────────
            c1, c2, c3 = st.columns(3)
            c1.markdown(kpi_card("Monthly EMI",    fmt_inr(int(emi)),        "Per month"),      unsafe_allow_html=True)
            c2.markdown(kpi_card("Total Interest", fmt_inr(int(total_int)),  f"Over {tenure_yr} yrs"), unsafe_allow_html=True)
            c3.markdown(kpi_card("Total Payment",  fmt_inr(int(total_pay)),  "Principal + Interest"), unsafe_allow_html=True)

            st.markdown("<br>", unsafe_allow_html=True)
            c4, c5 = st.columns(2)
            c4.markdown(kpi_card("EMI / Income Ratio", f"{emi_ratio:.1f}%",
                color=eli_color), unsafe_allow_html=True)
            c5.markdown(kpi_card("Loan Eligibility", eligibility,
                color=eli_color), unsafe_allow_html=True)

            st.markdown("<br>", unsafe_allow_html=True)

            # ── Payment Breakup Pie ───────────────────────────
            st.markdown("#### 🥧 Payment Breakup")
            fig_pie = px.pie(
                names=["Principal Amount", "Total Interest"],
                values=[int(loan_amt), int(total_int)],
                hole=0.45,
                color_discrete_sequence=["#1a73e8", "#f78166"]
            )
            fig_pie.update_layout(**PLOTLY_LAYOUT, height=280,
                legend=dict(bgcolor="#ffffff", bordercolor="#dddddd",
                            font=dict(color="#1a1a1a")))
            st.plotly_chart(fig_pie, use_container_width=True)

            # ── Amortization Schedule ─────────────────────────
            balance  = loan_amt
            yearly   = {}
            for month in range(1, n + 1):
                int_part  = balance * r
                prin_part = emi - int_part
                balance   = max(0, balance - prin_part)
                yr        = (month - 1) // 12 + 1
                if yr not in yearly:
                    yearly[yr] = {"Year": yr, "Principal": 0.0,
                                  "Interest": 0.0, "Balance": 0.0}
                yearly[yr]["Principal"] += prin_part
                yearly[yr]["Interest"]  += int_part
                yearly[yr]["Balance"]    = balance

            amo_df = pd.DataFrame(list(yearly.values()))
            amo_df["Principal"] = amo_df["Principal"].astype(int)
            amo_df["Interest"]  = amo_df["Interest"].astype(int)
            amo_df["Balance"]   = amo_df["Balance"].astype(int)


            # ── Amortization Table ────────────────────────────
            st.markdown("#### 📋 Year-by-Year Breakdown Table")
            amo_display = amo_df.copy()
            amo_display["Principal Paid"] = amo_display["Principal"].apply(fmt_inr)
            amo_display["Interest Paid"]  = amo_display["Interest"].apply(fmt_inr)
            amo_display["Balance Left"]   = amo_display["Balance"].apply(fmt_inr)
            st.dataframe(
                amo_display[["Year","Principal Paid","Interest Paid","Balance Left"]],
                use_container_width=True, hide_index=True
            )

            # ── Bank Rate Comparison Table ────────────────────
            st.markdown("#### 🏦 Compare All Banks for Your Loan")
            cmp_rows = []
            for bk, rt in bank_rates.items():
                if rt is None:
                    continue
                rr  = rt / 100 / 12
                em  = loan_amt * rr * (1+rr)**n / ((1+rr)**n - 1)
                tot = em * n
                cmp_rows.append({
                    "Bank":           bk.split(" (")[0],
                    "Interest Rate":  f"{rt}%",
                    "Monthly EMI":    fmt_inr(int(em)),
                    "Total Interest": fmt_inr(int(tot - loan_amt)),
                    "Total Payment":  fmt_inr(int(tot)),
                    "Savings vs Max": fmt_inr(int(max(b["Monthly EMI"] for b in
                                        [{"Monthly EMI": loan_amt * (br/100/12) *
                                          (1+br/100/12)**n / ((1+br/100/12)**n-1)}
                                         for br in bank_rates.values() if br]) * n - tot))
                                        if len(cmp_rows) > 0 else "—"
                })
            st.dataframe(pd.DataFrame(cmp_rows), use_container_width=True, hide_index=True)

        else:
            st.markdown("""
            <div class="res-card" style="text-align:center;padding:80px">
              <div style="font-size:3rem">💸</div>
              <div style="font-size:1.15rem;font-weight:600;color:#1a73e8;margin:14px 0">
                Home Loan EMI Calculator
              </div>
              <div style="font-size:.85rem;color:#8b949e">
                Enter your property price and loan details<br>
                on the left, then click Calculate EMI
              </div>
            </div>""", unsafe_allow_html=True)
# ══════════════════════════════════════════════════════════════
#  PAGE 10 — BLUEPRINT GENERATOR
# ══════════════════════════════════════════════════════════════
elif page == "🏗️ Blueprint Generator":
    sec_header("🏗️", "Blueprint Generator",
               "Engineering grid blueprint · Furniture · PDF Download · Side-by-side Compare")

    import matplotlib.pyplot as plt
    import matplotlib.patches as patches
    from matplotlib.patches import FancyBboxPatch
    from matplotlib.lines import Line2D
    import io

    ROOM_COLORS = {
        "Living Room":    "#C8E6FA",
        "Master Bedroom": "#C8F0D8",
        "Bedroom 2":      "#FFE0B2",
        "Bedroom 3":      "#FFCDD2",
        "Bedroom 4":      "#E1BEE7",
        "Bedroom 5":      "#B2EBF2",
        "Master Bath":    "#D1C4E9",
        "Bathroom 2":     "#CFD8DC",
        "Bathroom 3":     "#CFD8DC",
        "Kitchen":        "#FFF9C4",
        "Dining":         "#DCEDC8",
        "Balcony":        "#B2DFDB",
        "Study/Pooja":    "#FFE0B2",
        "Garage":         "#ECEFF1",
    }

    INTERIOR_COLORS = {
        "Living Room":    {"Walls":"Ivory White","Ceiling":"Pure White","Floor":"Golden Oak Wood","Accent":"Ocean Blue"},
        "Master Bedroom": {"Walls":"Mint Cream","Ceiling":"Pure White","Floor":"Walnut Brown","Accent":"Forest Green"},
        "Bedroom 2":      {"Walls":"Linen White","Ceiling":"Pure White","Floor":"Chestnut Wood","Accent":"Burnt Orange"},
        "Bedroom 3":      {"Walls":"Blush Pink","Ceiling":"Pure White","Floor":"Mahogany","Accent":"Deep Red"},
        "Bedroom 4":      {"Walls":"Sky Blue","Ceiling":"Pure White","Floor":"Teak Wood","Accent":"Navy Blue"},
        "Bedroom 5":      {"Walls":"Lavender Mist","Ceiling":"Pure White","Floor":"Pine Wood","Accent":"Violet"},
        "Kitchen":        {"Walls":"Cream Yellow","Ceiling":"Pure White","Floor":"Grey Slate Tile","Accent":"Mustard Yellow"},
        "Master Bath":    {"Walls":"Lilac White","Ceiling":"Pure White","Floor":"Charcoal Tile","Accent":"Deep Purple"},
        "Bathroom 2":     {"Walls":"Cool Grey","Ceiling":"Pure White","Floor":"Dark Slate","Accent":"Steel Blue"},
        "Dining":         {"Walls":"Sage Green","Ceiling":"Pure White","Floor":"Dark Walnut","Accent":"Olive Green"},
        "Balcony":        {"Walls":"Open Air","Ceiling":"Open Sky","Floor":"Stone Grey Tile","Accent":"Teal Green"},
        "Study/Pooja":    {"Walls":"Warm Beige","Ceiling":"Pure White","Floor":"Sandalwood","Accent":"Saffron Orange"},
        "Garage":         {"Walls":"Off White","Ceiling":"Light Grey","Floor":"Cement Grey","Accent":"Charcoal"},
    }

    INTERIOR_HEX = {
        "Living Room":    {"Walls":"#F5F0E8","Ceiling":"#FFFFFF","Floor":"#D4A96A","Accent":"#2C5F8A"},
        "Master Bedroom": {"Walls":"#E8F4F0","Ceiling":"#FFFFFF","Floor":"#8B6914","Accent":"#1A6B3C"},
        "Bedroom 2":      {"Walls":"#FDF5E6","Ceiling":"#FFFFFF","Floor":"#A0522D","Accent":"#E67E22"},
        "Bedroom 3":      {"Walls":"#FDE8E8","Ceiling":"#FFFFFF","Floor":"#8B4513","Accent":"#C0392B"},
        "Bedroom 4":      {"Walls":"#EAF6FF","Ceiling":"#FFFFFF","Floor":"#7B5E3A","Accent":"#1A3A5C"},
        "Bedroom 5":      {"Walls":"#F3E8FD","Ceiling":"#FFFFFF","Floor":"#8B7355","Accent":"#6A1B9A"},
        "Kitchen":        {"Walls":"#FFFDE7","Ceiling":"#FFFFFF","Floor":"#607D8B","Accent":"#F39C12"},
        "Master Bath":    {"Walls":"#EDE7F6","Ceiling":"#FFFFFF","Floor":"#455A64","Accent":"#7B1FA2"},
        "Bathroom 2":     {"Walls":"#ECEFF1","Ceiling":"#FFFFFF","Floor":"#546E7A","Accent":"#37474F"},
        "Dining":         {"Walls":"#F1F8E9","Ceiling":"#FFFFFF","Floor":"#795548","Accent":"#558B2F"},
        "Balcony":        {"Walls":"#E0F2F1","Ceiling":"#87CEEB","Floor":"#78909C","Accent":"#00897B"},
        "Study/Pooja":    {"Walls":"#FFF3E0","Ceiling":"#FFFFFF","Floor":"#6D4C41","Accent":"#E65100"},
        "Garage":         {"Walls":"#ECEFF1","Ceiling":"#B0BEC5","Floor":"#546E7A","Accent":"#37474F"},
    }

    FURNITURE = {
        "Living Room":    [("Sofa",0.35,0.15,"#8B7355"),("Coffee Table",0.45,0.38,"#A0522D"),
                           ("TV Unit",0.35,0.75,"#4A4A4A"),("Chair",0.72,0.25,"#8B7355")],
        "Master Bedroom": [("Bed",0.30,0.25,"#8B6914"),("Wardrobe",0.75,0.15,"#5C4827"),
                           ("Dresser",0.75,0.65,"#7B5E3A"),("Side Table",0.55,0.20,"#A0896B")],
        "Bedroom 2":      [("Bed",0.30,0.30,"#CD853F"),("Wardrobe",0.75,0.15,"#8B6914"),
                           ("Study Desk",0.70,0.65,"#A0522D")],
        "Bedroom 3":      [("Bed",0.30,0.30,"#BC8F5F"),("Wardrobe",0.75,0.15,"#8B6914")],
        "Bedroom 4":      [("Bed",0.30,0.30,"#BC8F5F"),("Wardrobe",0.75,0.15,"#8B6914")],
        "Bedroom 5":      [("Bed",0.30,0.30,"#BC8F5F"),("Wardrobe",0.75,0.15,"#8B6914")],
        "Kitchen":        [("Counter",0.15,0.50,"#B0BEC5"),("Stove",0.15,0.20,"#607D8B"),
                           ("Sink",0.15,0.75,"#90A4AE"),("Fridge",0.75,0.15,"#CFD8DC")],
        "Master Bath":    [("Toilet",0.20,0.20,"#ECEFF1"),("Sink",0.75,0.20,"#ECEFF1"),
                           ("Shower",0.55,0.65,"#B2EBF2")],
        "Bathroom 2":     [("Toilet",0.20,0.20,"#ECEFF1"),("Sink",0.75,0.20,"#ECEFF1")],
        "Dining":         [("Table",0.40,0.40,"#8B6914"),("Chair",0.20,0.30,"#6D4C41"),
                           ("Chair",0.20,0.55,"#6D4C41"),("Chair",0.65,0.30,"#6D4C41"),
                           ("Chair",0.65,0.55,"#6D4C41")],
        "Balcony":        [("Plant",0.20,0.20,"#2E7D32"),("Chair",0.60,0.50,"#795548")],
        "Study/Pooja":    [("Desk",0.25,0.40,"#8D6E63"),("Chair",0.50,0.55,"#6D4C41"),
                           ("Shelf",0.75,0.25,"#5D4037")],
    }

    def calc_rooms(area, bedrooms, bathrooms, hall, kitchen,
                   dining, balcony, study, garage):
        ratios = {}
        if hall:    ratios["Living Room"]  = 0.18
        for i in range(bedrooms):
            lbl = "Master Bedroom" if i==0 else f"Bedroom {i+1}"
            ratios[lbl] = 0.14 if i==0 else 0.11
        for i in range(bathrooms):
            lbl = "Master Bath" if i==0 else f"Bathroom {i+1}"
            ratios[lbl] = 0.06
        if kitchen: ratios["Kitchen"]     = 0.10
        if dining:  ratios["Dining"]      = 0.09
        if balcony: ratios["Balcony"]     = 0.05
        if study:   ratios["Study/Pooja"] = 0.07
        if garage:  ratios["Garage"]      = 0.10
        total = sum(ratios.values())
        return {k: round(v/total*area) for k,v in ratios.items()}

    def split_columns(room_areas):
        items = list(room_areas.items())
        left, right = [], []
        tl = tr = 0
        for name, area in items:
            if tl <= tr:
                left.append((name, area)); tl += area
            else:
                right.append((name, area)); tr += area
        return left, right

    def draw_2d_blueprint(room_areas, orient, area, bedrooms,
                          show_furniture=True, title_suffix=""):
        # large figure for readable text
        fig, ax = plt.subplots(figsize=(20, 18))
        ax.set_facecolor("#E8F0F8")
        fig.patch.set_facecolor("#D0DCF0")
        ax.set_xlim(0, 100)
        ax.set_ylim(0, 100)
        ax.set_aspect("equal")
        ax.axis("off")

        # grid lines
        for x in range(0, 101, 5):
            lw  = 0.8 if x % 10 == 0 else 0.3
            col = "#A8C4E0" if x % 10 == 0 else "#C4D8F0"
            ax.axvline(x, color=col, linewidth=lw, zorder=0)
        for y in range(0, 101, 5):
            lw  = 0.8 if y % 10 == 0 else 0.3
            col = "#A8C4E0" if y % 10 == 0 else "#C4D8F0"
            ax.axhline(y, color=col, linewidth=lw, zorder=0)

        # title bar
        ax.add_patch(patches.Rectangle(
            (0, 94), 100, 6,
            facecolor="#1A3A5C", edgecolor="none", zorder=5))
        ax.text(50, 97,
                f"FLOOR PLAN {title_suffix}  —  {area} SQ.FT  |  "
                f"{bedrooms}BHK  |  {orient.upper()} FACING",
                ha="center", va="center",
                fontsize=22, fontweight="bold",
                color="white", zorder=6)

        # outer wall
        ax.add_patch(FancyBboxPatch(
            (2, 2), 96, 91,
            boxstyle="square,pad=0",
            linewidth=5, edgecolor="#1A3A5C",
            facecolor="none", zorder=3))

        left, right = split_columns(room_areas)
        lt = sum(a for _, a in left)  or 1
        rt = sum(a for _, a in right) or 1

        def draw_col(rooms_col, x0, x1, total):
            y = 3
            h_total = 89
            for name, rm_area in rooms_col:
                rh = max(10, rm_area / total * h_total)
                rw = x1 - x0
                color = ROOM_COLORS.get(name, "#E3EFF8")

                # room fill
                ax.add_patch(FancyBboxPatch(
                    (x0+0.4, y+0.4), rw-0.8, rh-0.8,
                    boxstyle="square,pad=0",
                    linewidth=2.5, edgecolor="#1A3A5C",
                    facecolor=color, zorder=2))

                # hatch wet areas
                if "Bath" in name or "Kitchen" in name:
                    ax.add_patch(FancyBboxPatch(
                        (x0+0.4, y+0.4), rw-0.8, rh-0.8,
                        boxstyle="square,pad=0",
                        linewidth=0, edgecolor="none",
                        facecolor="none",
                        hatch="///", zorder=2, alpha=0.2))

                # room name — big and bold
                ax.text(x0 + rw/2, y + rh/2 + 2.2,
                        name,
                        ha="center", va="center",
                        fontsize=12, fontweight="bold",
                        color="#1A3A5C", zorder=4)

                # area text
                ax.text(x0 + rw/2, y + rh/2 - 1.5,
                        f"{rm_area} sqft",
                        ha="center", va="center",
                        fontsize=10, color="#2C5F8A",
                        zorder=4)

                # dimension line bottom
                ax.annotate("",
                    xy=(x0+rw-1.5, y+1.5),
                    xytext=(x0+1.5, y+1.5),
                    arrowprops=dict(
                        arrowstyle="<->",
                        color="#1A3A5C", lw=1.2),
                    zorder=5)
                ax.text(x0 + rw/2, y + 3.2,
                        f"{round(rw*0.3, 1)} m",
                        ha="center", fontsize=8,
                        color="#1A3A5C", fontweight="bold",
                        zorder=5)

                # door arc
                if "Bath" not in name and "Balcony" not in name:
                    arc = patches.Arc(
                        (x0+3, y+0.4), 5, 5,
                        angle=0, theta1=0, theta2=90,
                        color="#1A3A5C", lw=1.8, zorder=5)
                    ax.add_patch(arc)
                    ax.plot([x0+3, x0+3],
                            [y+0.4, y+2.9],
                            color="#1A3A5C", lw=1.8, zorder=5)

                # window dashes
                if any(r in name for r in
                       ["Bedroom","Living","Dining","Study","Balcony"]):
                    wx = [x0+rw*0.25, x0+rw*0.75]
                    wy = [y+rh-0.6,   y+rh-0.6]
                    ax.plot(wx, wy, color="#2196F3",
                            lw=3.5, ls="--", zorder=5)
                    ax.plot(wx,
                            [y+rh-1.8, y+rh-1.8],
                            color="#2196F3", lw=1.2, zorder=5)

                # furniture
                if show_furniture and name in FURNITURE:
                    for fname, fx, fy, fcol in FURNITURE[name]:
                        px = x0 + rw*0.08 + fx*(rw*0.84)
                        py = y  + rh*0.08 + fy*(rh*0.84)
                        fw = rw*0.12
                        fh = rh*0.12
                        ax.add_patch(FancyBboxPatch(
                            (px-fw/2, py-fh/2), fw, fh,
                            boxstyle="round,pad=0.3",
                            linewidth=1.0,
                            edgecolor="#333333",
                            facecolor=fcol,
                            alpha=0.88, zorder=3))
                        ax.text(px, py, fname[:5],
                                ha="center", va="center",
                                fontsize=6, color="white",
                                fontweight="bold", zorder=4)
                y += rh

        draw_col(left,  2,  51, lt)
        draw_col(right, 51, 98, rt)

        # center divider wall
        ax.plot([50, 50], [3, 93],
                color="#1A3A5C", lw=2.5, zorder=3)

        # entrance arrow
        EDIR = {
            "North": (50, 93.5, "↑  MAIN ENTRANCE"),
            "South": (50, 2.5,  "↓  MAIN ENTRANCE"),
            "East":  (97, 50,   "→  MAIN ENTRANCE"),
            "West":  (3,  50,   "←  MAIN ENTRANCE"),
        }
        ex, ey, elbl = EDIR[orient]
        ax.text(ex, ey, elbl,
                ha="center", va="center",
                fontsize=10, fontweight="bold",
                color="white", zorder=7,
                bbox=dict(boxstyle="round,pad=0.5",
                          facecolor="#E74C3C",
                          edgecolor="white", lw=2))

        # compass
        ax.text(93, 7, "N\n↑",
                ha="center", va="center",
                fontsize=11, fontweight="bold",
                color="#1A3A5C", zorder=7,
                bbox=dict(boxstyle="circle,pad=0.6",
                          facecolor="white",
                          edgecolor="#1A3A5C", lw=2))

        # legend
        legend_handles = [
            Line2D([0],[0], color="#2196F3", lw=2.5,
                   ls="--", label="Window"),
            patches.Patch(facecolor="white",
                          edgecolor="#1A3A5C",
                          label="Door arc"),
            patches.Patch(facecolor="#E8F0F8",
                          hatch="///",
                          edgecolor="#555",
                          label="Wet area", alpha=0.5),
        ]
        ax.legend(handles=legend_handles,
                  loc="lower left",
                  fontsize=9,
                  framealpha=0.95,
                  edgecolor="#1A3A5C",
                  facecolor="white")

        # scale bar
        ax.plot([68, 80], [5, 5],
                color="#1A3A5C", lw=2.5)
        ax.plot([68, 68], [4, 6],
                color="#1A3A5C", lw=2)
        ax.plot([80, 80], [4, 6],
                color="#1A3A5C", lw=2)
        ax.text(74, 7, "Scale: 3 m",
                ha="center", fontsize=9,
                color="#1A3A5C", fontweight="bold")

        plt.tight_layout(pad=0.5)
        return fig

    def fig_to_bytes(fig, dpi=180):
        buf = io.BytesIO()
        fig.savefig(buf, format="png", dpi=dpi,
                    bbox_inches="tight")
        buf.seek(0)
        return buf.read()

    def make_pdf_download(img_bytes_list, labels):
        from reportlab.lib.pagesizes import A4, landscape
        from reportlab.platypus import (SimpleDocTemplate,
                                        Image as RLImage,
                                        Paragraph, Spacer)
        from reportlab.lib.styles import getSampleStyleSheet
        from reportlab.lib.units import cm
        buf = io.BytesIO()
        doc = SimpleDocTemplate(buf, pagesize=landscape(A4),
                                leftMargin=1*cm, rightMargin=1*cm,
                                topMargin=1*cm, bottomMargin=1*cm)
        styles = getSampleStyleSheet()
        story  = []
        for img_bytes, label in zip(img_bytes_list, labels):
            story.append(Paragraph(label, styles["Heading2"]))
            story.append(Spacer(1, 0.3*cm))
            img_buf = io.BytesIO(img_bytes)
            story.append(RLImage(img_buf,
                                 width=25*cm, height=18*cm))
            story.append(Spacer(1, 0.5*cm))
        doc.build(story)
        buf.seek(0)
        return buf.read()

    # ── UI inputs ─────────────────────────────────────────────
    st.markdown("### 🏠 Configure Floor Plans")
    tab_a, tab_b = st.tabs(["📐 Plan A", "📐 Plan B (Compare)"])

    def plan_inputs(key):
        c1, c2 = st.columns(2)
        with c1:
            area      = st.slider("Total Area (sqft)", 400, 5000, 1200, 50, key=f"area_{key}")
            bedrooms  = st.selectbox("Bedrooms", [1,2,3,4,5], index=1, key=f"bed_{key}")
            bathrooms = st.selectbox("Bathrooms", [1,2,3], index=1, key=f"bat_{key}")
            orient    = st.selectbox("Facing", ["North","South","East","West"], key=f"ori_{key}")
        with c2:
            hall    = st.checkbox("Living / Hall",  value=True,  key=f"hall_{key}")
            kitchen = st.checkbox("Kitchen",        value=True,  key=f"kit_{key}")
            dining  = st.checkbox("Dining Area",    value=False, key=f"din_{key}")
            balcony = st.checkbox("Balcony",        value=True,  key=f"bal_{key}")
            study   = st.checkbox("Study / Pooja",  value=False, key=f"stu_{key}")
            garage  = st.checkbox("Garage",         value=False, key=f"gar_{key}")
        return dict(area=area, bedrooms=bedrooms, bathrooms=bathrooms,
                    orient=orient, hall=hall, kitchen=kitchen,
                    dining=dining, balcony=balcony,
                    study=study, garage=garage)

    with tab_a:
        pa = plan_inputs("a")
    with tab_b:
        enable_b = st.checkbox("Enable Plan B for comparison", value=False)
        pb = plan_inputs("b") if enable_b else None

    show_furn = st.checkbox("🪑 Show Furniture inside rooms", value=True)
    gen_btn   = st.button("🏗️ Generate Blueprint")

    if gen_btn:
        with st.spinner("Generating blueprint…"):
            ra = calc_rooms(pa["area"], pa["bedrooms"], pa["bathrooms"],
                            pa["hall"], pa["kitchen"], pa["dining"],
                            pa["balcony"], pa["study"], pa["garage"])
            fig_2d_a = draw_2d_blueprint(
                ra, pa["orient"], pa["area"],
                pa["bedrooms"], show_furn, "A")

            if enable_b and pb:
                rb = calc_rooms(pb["area"], pb["bedrooms"], pb["bathrooms"],
                                pb["hall"], pb["kitchen"], pb["dining"],
                                pb["balcony"], pb["study"], pb["garage"])
                fig_2d_b = draw_2d_blueprint(
                    rb, pb["orient"], pb["area"],
                    pb["bedrooms"], show_furn, "B")

        # ── Blueprint display ─────────────────────────────────
        st.markdown("---")
        st.markdown("### 📐 Engineering Grid Blueprint")

        if enable_b and pb:
            c2d_a, c2d_b = st.columns(2)
            with c2d_a:
                st.markdown("#### Plan A")
                buf = io.BytesIO()
                fig_2d_a.savefig(buf, format="png",
                                 dpi=180, bbox_inches="tight")
                buf.seek(0)
                st.image(buf, use_container_width=True)
                plt.close(fig_2d_a)
            with c2d_b:
                st.markdown("#### Plan B")
                buf = io.BytesIO()
                fig_2d_b.savefig(buf, format="png",
                                 dpi=180, bbox_inches="tight")
                buf.seek(0)
                st.image(buf, use_container_width=True)
                plt.close(fig_2d_b)
        else:
            buf = io.BytesIO()
            fig_2d_a.savefig(buf, format="png",
                             dpi=200, bbox_inches="tight")
            buf.seek(0)
            st.image(buf, use_container_width=True)
            plt.close(fig_2d_a)

        # ── Room breakdown table ──────────────────────────────
        st.markdown("---")
        st.markdown("### 📋 Room Area Breakdown")
        if enable_b and pb:
            ct_a, ct_b = st.columns(2)
            with ct_a:
                st.markdown("**Plan A**")
                dfa = pd.DataFrame([{
                    "Room": name,
                    "Area (sqft)": a,
                    "% of Total": f"{a/pa['area']*100:.1f}%",
                    "Dimensions": f"≈ {round((a**0.5)*0.9,1)} × {round((a**0.5)*1.1,1)} ft"
                } for name, a in ra.items()])
                st.dataframe(dfa, use_container_width=True, hide_index=True)
            with ct_b:
                st.markdown("**Plan B**")
                dfb = pd.DataFrame([{
                    "Room": name,
                    "Area (sqft)": a,
                    "% of Total": f"{a/pb['area']*100:.1f}%",
                    "Dimensions": f"≈ {round((a**0.5)*0.9,1)} × {round((a**0.5)*1.1,1)} ft"
                } for name, a in rb.items()])
                st.dataframe(dfb, use_container_width=True, hide_index=True)
        else:
            dfa = pd.DataFrame([{
                "Room": name,
                "Area (sqft)": a,
                "% of Total": f"{a/pa['area']*100:.1f}%",
                "Dimensions": f"≈ {round((a**0.5)*0.9,1)} × {round((a**0.5)*1.1,1)} ft"
            } for name, a in ra.items()])
            st.dataframe(dfa, use_container_width=True, hide_index=True)

        # ── Interior color suggestions ────────────────────────
        st.markdown("---")
        st.markdown("### 🎨 Interior Design Color Suggestions")
        rooms_to_show = ra
        ic_cols = st.columns(3)
        for idx, (room, _) in enumerate(rooms_to_show.items()):
            names  = INTERIOR_COLORS.get(room, {
                "Walls":"Off White","Ceiling":"Pure White",
                "Floor":"Teak Wood","Accent":"Navy Blue"})
            hexes  = INTERIOR_HEX.get(room, {
                "Walls":"#F5F5F5","Ceiling":"#FFFFFF",
                "Floor":"#8B7355","Accent":"#1A73E8"})
            with ic_cols[idx % 3]:
                swatches = "".join([
                    f'<div style="display:inline-flex;align-items:center;'
                    f'margin:4px 6px 4px 0;">'
                    f'<div style="width:26px;height:26px;border-radius:50%;'
                    f'background:{hexes[k]};border:2px solid #ccc;'
                    f'margin-right:6px;flex-shrink:0"></div>'
                    f'<span style="font-size:.75rem;color:#333">'
                    f'<b>{k}:</b> {v}</span></div>'
                    for k, v in names.items()
                ])
                st.markdown(f"""
                <div class="res-card" style="padding:14px;margin-bottom:10px">
                  <div style="font-weight:700;font-size:.9rem;
                              color:#1a3a5c;margin-bottom:10px;
                              border-bottom:2px solid #1a73e8;
                              padding-bottom:5px">{room}</div>
                  {swatches}
                </div>""", unsafe_allow_html=True)

        # ── Vastu tips ────────────────────────────────────────
        st.markdown("---")
        st.markdown("### 🧿 Vastu Analysis")
        vastu_ok  = pa["orient"] in ["North","East"]
        vastu_msg = ("✅ North / East facing — excellent for natural light and positive energy."
                     if vastu_ok else
                     f"ℹ️ {pa['orient']} facing — ensure South-West master bedroom "
                     f"and South-East kitchen for Vastu compliance.")
        st.markdown(f"""
        <div class="res-card" style="background:#{'e8f5e9' if vastu_ok else 'fff8e1'};
             border-color:#{'2e7d32' if vastu_ok else 'f57f17'}">
          <b style="color:#{'2e7d32' if vastu_ok else 'e65100'}">🧿 Vastu Score</b>
          <div style="font-size:.9rem;color:#1a1a1a;margin-top:8px">{vastu_msg}</div>
          <ul style="font-size:.85rem;color:#333;line-height:2.2;margin-top:8px">
            <li>Master Bedroom → <b>South-West</b> corner (stability & rest)</li>
            <li>Kitchen → <b>South-East</b> corner (fire element)</li>
            <li>Pooja / Study → <b>North-East</b> corner (positivity)</li>
            <li>Main Entrance → <b>North or East</b> (wealth & health)</li>
            <li>Garage → <b>North-West</b> or South-East (movement energy)</li>
          </ul>
        </div>""", unsafe_allow_html=True)

        # ── PDF Download ──────────────────────────────────────
        st.markdown("---")
        st.markdown("### 📥 Download Blueprint as PDF")
        with st.spinner("Preparing PDF…"):
            img_list, lbl_list = [], []
            fig_pdf_a = draw_2d_blueprint(
                ra, pa["orient"], pa["area"],
                pa["bedrooms"], show_furn, "A")
            img_list.append(fig_to_bytes(fig_pdf_a, dpi=200))
            lbl_list.append(
                f"Plan A — Engineering Blueprint ({pa['area']} sqft, "
                f"{pa['bedrooms']}BHK, {pa['orient']} Facing)")
            plt.close(fig_pdf_a)

            if enable_b and pb:
                fig_pdf_b = draw_2d_blueprint(
                    rb, pb["orient"], pb["area"],
                    pb["bedrooms"], show_furn, "B")
                img_list.append(fig_to_bytes(fig_pdf_b, dpi=200))
                lbl_list.append(
                    f"Plan B — Engineering Blueprint ({pb['area']} sqft, "
                    f"{pb['bedrooms']}BHK, {pb['orient']} Facing)")
                plt.close(fig_pdf_b)

            pdf_bytes = make_pdf_download(img_list, lbl_list)

        st.download_button(
            label="📥 Download Full Blueprint PDF",
            data=pdf_bytes,
            file_name="propaura_blueprint.pdf",
            mime="application/pdf"
        )

    else:
        st.markdown("""
        <div class="res-card" style="text-align:center;padding:70px">
          <div style="font-size:3.5rem">🏗️</div>
          <div style="font-size:1.2rem;font-weight:700;
                      color:#1a73e8;margin:14px 0">
            Advanced Blueprint Generator
          </div>
          <div style="font-size:.88rem;color:#8b949e;line-height:2.2">
            📐 Engineering grid floor plan with large labels<br>
            🪑 Auto furniture placement per room<br>
            🎨 Interior color suggestions with real color names<br>
            📊 Side-by-side plan comparison<br>
            📥 Download as PDF
          </div>
        </div>""", unsafe_allow_html=True)
# ══════════════════════════════════════════════════════════════
#  FOOTER
# ══════════════════════════════════════════════════════════════
st.markdown("""
<hr>
<div style="text-align:center;color:#8b949e;font-size:.73rem;padding:8px 0">
  🏙️ <b style="color:#58a6ff">PropAura</b> — Tamil Nadu Market Intelligence &nbsp;|&nbsp;
  Final Year Data Science Project &nbsp;|&nbsp;
  Cities: Chennai · Coimbatore · Madurai · Trichy · Salem + 6 more &nbsp;|&nbsp;
  Powered by XGBoost + SHAP + Streamlit
</div>
""", unsafe_allow_html=True)



Writing app.py


In [ ]:
with open("app.py", "r") as f:
    code = f.read()

old = '''def kpi_card(label, val, sub="", color="#1a73e8"):
    sub_html = f\'<div style="font-size:.72rem;color:#2e7d32;margin-top:5px">{sub}</div>\' if sub else ""
    return (
        \'<div style="background:#f8f9fa;border:1px solid #dddddd;border-radius:12px;\' +
        \'padding:20px 22px;text-align:center;">\' +
        f\'<div style="font-size:.72rem;font-weight:700;color:#555555;text-transform:uppercase;\' +
        f\'letter-spacing:.9px;margin-bottom:6px">{label}</div>\' +
        f\'<div style="font-size:1.85rem;font-weight:800;color:{color};line-height:1.1">{val}</div>\' +
        sub_html +
        \'</div>\'
    )'''

new = '''def kpi_card(label, val, sub="", color="#1a73e8"):
    sub_part = f'<div style="font-size:.7rem;color:#2e7d32;margin-top:4px">{sub}</div>' if sub else ""
    return f\'\'\'<div style="background:#f8f9fa;border:1px solid #dddddd;border-radius:12px;
padding:16px;text-align:center;height:100%;word-wrap:break-word;overflow:hidden;">
<div style="font-size:.65rem;font-weight:700;color:#555;text-transform:uppercase;
letter-spacing:.8px;margin-bottom:8px;line-height:1.3">{label}</div>
<div style="font-size:1.5rem;font-weight:800;color:{color};line-height:1.1;
word-break:break-word">{val}</div>{sub_part}</div>\'\'\'  '''

code = code.replace(old, new)

with open("app.py", "w") as f:
    f.write(code)

print("✅ KPI fix done!")

✅ KPI fix done!


In [ ]:
with open("app.py", "r") as f:
    code = f.read()

# Fix plotly layout - white background, visible legend
code = code.replace(
    'paper_bgcolor="#1c2128", plot_bgcolor="#1c2128",\n    font_color="#e6edf3"',
    'paper_bgcolor="#ffffff", plot_bgcolor="#f8f9fa",\n    font_color="#1a1a1a"'
)
code = code.replace(
    'paper_bgcolor="#1c2128",font_color="#e6edf3"',
    'paper_bgcolor="#ffffff",font_color="#1a1a1a"'
)
code = code.replace(
    'paper_bgcolor="#1c2128", font_color="#e6edf3"',
    'paper_bgcolor="#ffffff", font_color="#1a1a1a"'
)

# Fix all legend backgrounds
code = code.replace(
    "legend=dict(bgcolor=\"#1c2128\",bordercolor=\"#30363d\")",
    "legend=dict(bgcolor=\"#ffffff\",bordercolor=\"#dddddd\",font=dict(color='#1a1a1a'))"
)

# Fix all axis colors
code = code.replace('color="#8b949e"', 'color="#333333"')
code = code.replace("color=\"#8b949e\"", "color=\"#333333\"")
code = code.replace('gridcolor="#30363d"', 'gridcolor="#eeeeee"')
code = code.replace("gridcolor=\"#30363d\"", "gridcolor=\"#eeeeee\"")

with open("app.py", "w") as f:
    f.write(code)

print("✅ Charts fix done!")

✅ Charts fix done!


In [ ]:
with open("app.py", "r") as f:
    code = f.read()

# Replace ALL dark map tiles with light OpenStreetMap
code = code.replace(
    'tiles="CartoDB dark_matter"',
    'tiles="OpenStreetMap"'
)
code = code.replace(
    "tiles=\"CartoDB dark_matter\"",
    "tiles=\"OpenStreetMap\""
)

# Fix mapbox heatmap to use light style
code = code.replace(
    'mapbox_style="carto-darkmatter"',
    'mapbox_style="open-street-map"'
)

# Fix all st_folium - stop blinking
code = code.replace(
    'st_folium(m, width=750, height=400)',
    'st_folium(m, width=750, height=400, returned_objects=[])'
)
code = code.replace(
    'st_folium(m, width=1050, height=450)',
    'st_folium(m, width=1050, height=450, returned_objects=[])'
)
code = code.replace(
    'st_folium(m, width=700, height=380)',
    'st_folium(m, width=700, height=380, returned_objects=[])'
)

with open("app.py", "w") as f:
    f.write(code)
print("✅ Maps fix done!")

✅ Maps fix done!


In [ ]:
with open("app.py", "r") as f:
    code = f.read()

# Fix multiselect tag CSS
code = code.replace(
    '''[data-baseweb="tag"] {
    background-color: #1a73e8 !important;
}
[data-baseweb="tag"] span {
    color: #ffffff !important;
}''',
    '''[data-baseweb="tag"] {
    background-color: #1a73e8 !important;
    border-radius: 4px !important;
}
[data-baseweb="tag"] span,
[data-baseweb="tag"] div,
[data-baseweb="tag"] * {
    color: #ffffff !important;
    font-weight: 600 !important;
}'''
)

with open("app.py", "w") as f:
    f.write(code)

print("✅ Multiselect fix done!")

✅ Multiselect fix done!


In [ ]:
import subprocess, time, re, os

# ✅ Kill any previous session
os.system("pkill -f streamlit 2>/dev/null")
os.system("pkill -f cloudflared 2>/dev/null")
time.sleep(3)

# ✅ Install cloudflared
os.system("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared")
os.system("chmod +x cloudflared")

# ✅ Start Streamlit with session fix flags
proc = subprocess.Popen(
    ["streamlit", "run", "app.py",
     "--server.port=8501",
     "--server.headless=true",
     "--server.enableCORS=false",
     "--server.enableXsrfProtection=false",
     "--server.enableWebsocketCompression=false",
     "--browser.gatherUsageStats=false",
     "--server.maxUploadSize=200",
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

print("⏳ Starting Streamlit... waiting 8 seconds")
time.sleep(8)

# ✅ Start Cloudflare tunnel
tunnel_proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8501"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

print("⏳ Starting tunnel... waiting 12 seconds")
time.sleep(12)

# ✅ Grab URL
for _ in range(30):
    line = tunnel_proc.stderr.readline().decode("utf-8")
    if not line:
        break
    match = re.search(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com", line)
    if match:
        url = match.group(0)
        print("=" * 60)
        print("🚀 Propaura AI is LIVE!")
        print(f"🌐 URL: {url}")
        print("=" * 60)
        break
else:
    print("⚠️ URL not found. Try running this cell again.")

⏳ Starting Streamlit... waiting 8 seconds
⏳ Starting tunnel... waiting 12 seconds
🚀 Propaura AI is LIVE!
🌐 URL: https://basket-seminars-alone-avon.trycloudflare.com
